# 09 — ACDC / MRzero: one voxel in z, dephasing along all three axes

Based on `06_acdc_echo32_b0_s0_t2star_MRzeroV3.ipynb`. The inputs, 32-echo fit,
orientation, hardware limits and measured GRE target are retained.
One voxel represents the entire slice thickness: **no subdivision into eight sub-slices**.
The ACDC x/y/z gradients come from the 3D calibration, with checks at all eight corners.
The measured B0 map is 2D: its gradient along the slice normal is assumed to be zero.
No z B0 gradient is inferred from the residual echo dispersion.

Restart the kernel before switching between generations. The functions used by the notebooks
are in this folder; acquisitions and calibration remain in the parent folder.
A cell immediately after the first simulation saves its parameters and its comparison
with the actual EPI scan, **before** optional baseline calibration.

**09 — B0 and dephasing inside MRzero**: the PDG is recomputed and executed
at every current evaluation. The filtered ACDC field is included in the B0 history
passed to the simulator. The physical voxel size and MRzero dephasing kernels are used;
no ACDC phase or sinc integral is added to the signal after simulation.

The standard MRzero API describes a static B0 map and a shared spatial dephasing kernel.
The local adapter extends its execution to B0 histories and local gradients for each voxel.
For the independent ideal GRE excitation histories inherited from 06, with no diffusion,
`B0_effective(t) = B0 + integral(ACDC_field, RF→t)/(t−RF)` preserves the phase exactly
at each ADC time, including the hardware response. It does not replace a general
multi-RF/refocused simulator. Residual dispersion without an identified spatial direction
remains independent of the currents and is evaluated by native kernels inside the simulator.
09 is slower than 08 because native magnetization is no longer cached between iterations.


In [ ]:
from pathlib import Path
from copy import deepcopy
from datetime import datetime
import hashlib, json, sys, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown

# Locate the project from the notebook folder or project folder.
PROJECT_ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents]
                   if (p/"data/ACDC_3T.npz").is_file() and (p/"Experiment").is_dir()),None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the notebook in the ACDC project folder.")
HELPER_DIR=PROJECT_ROOT/"MRzero_generations_08_09"
sys.path.insert(0,str(HELPER_DIR))
from twin_inputs import read_twin_inputs,prepare_echo_object
from voxel_3d import prepare_voxel_inputs, FiniteVoxel3DACDCModel
from multiecho_object import prepare_complex_echo_model,multiecho_comparison
from reference_echo_prior import apply_reference_amplitude_prior
from measured_image_target import configure_measured_gre_target,relative_image_mse
from b0_regularization import regularize_b0_local_plane
from inplane_geometry import inplane_matrix,FrameReconstruction
from twin_geometry import compare_magnitude
from acdc_gpu_field import solve_field_gpu
from acdc_gpu import response_matrices,select_device
from experiment_optimization import optimize_mrzero_image
from field_guidance import FieldGuidance
from live_optimization import LiveOptimizationPlot
from optimization_image_history import OptimizationImageHistory
from acdc_waveform_plot import plot_acdc_waveform
from measured_scan_comparison import compare_measured_scan_set
from export_acdc_waveform import quantize_currents,export_waveform,export_zero_time_copy,validate_waveform_settings
from shim_export_timing import sequence_trigger_timing,waveform_edges_for_origin
from field_shim_export import export_field_shim_waveforms
from prepare_acdc_inputs import read_scan
from read_acdc_b0 import read_b0_sequence
from read_acdc_sequence import read_sequence

from first_simulation_report import first_simulation_report
from native_b0_model import NativeB0ACDCModel

GENERATION="09"
NOTEBOOK_NAME="09_acdc_echo32_b0_s0_t2star_MRzero.ipynb"
import continuous_reconstruction
continuous_reconstruction.FORCE_DIRECT_DFT=(GENERATION=="08_V2")
continuous_reconstruction.MATRIX_DFT_ENABLED=(GENERATION=="08_V2")
# Active configuration from 06 V3, without the old overwritten parameter blocks.
INPUT_DIR=PROJECT_ROOT/"acquisition_029_(06)"
GRE_EPI_SEQUENCE_NAME="gre_epi_128x128_fov200mm_nav3_cimax_snr_t2star40ms_line_triggers_v6.seq"
GRE_EPI_SCAN_PATH=None
REFERENCE_SCAN_PATH=None
REFERENCE_SEQUENCE_NAME="gre_128x128_fov200mm_flip15deg_tr15ms.seq"
B0_SEQUENCE_NAME="megre_b0_s0_t2star_128x128_fov200mm_32echo.seq"
B0_SCAN_PATH=None
CALIBRATION_PATH=PROJECT_ROOT/"data/ACDC_3T.npz"
OUTPUT_DIR=HELPER_DIR/"results"/datetime.now().strftime("mrzero09_%Y%m%d_%H%M%S_%f")
WAVEFORM_OUTPUT_DIR=INPUT_DIR/"waveform"
print("Calibration:",CALIBRATION_PATH.resolve())
print("Results:",OUTPUT_DIR)

## 1. Parameters to change

**Disabled channels**: `DISABLED_CHANNELS=[]` keeps every channel active.
For example, `[7]` disables physical output 7, and `[7, 12]` disables two outputs.
The numbers refer to `WAVEFORM_PARAMETERS["channel_map"]` (1 to 32 by default),
rather than Python indices; output 7 corresponds to `ch6_A` in the CSV.
Their currents are constrained to zero in the initial shim and throughout the image,
static and dynamic optimizations. The other channels are optimized accordingly.
The columns are retained, and export rejects nonzero current on a disabled channel.
After making a change, rerun from this cell to rebuild the model.

**Data → coil orientation**: `ALIGNMENT_ANGLE_DEG` sets the angle around Z,
in degrees, positive from X toward Y. `ALIGNMENT_MIRROR_X` replaces X with −X;
`ALIGNMENT_MIRROR_Y` replaces Y with −Y. Mirroring is applied **before** rotation.
The amplitude, B0 and ADC coordinates follow this orientation together. The coils remain
in their calibration frame, so changing this alignment changes the required currents.
The initial value of 90° follows the active configuration in 06 V3.

**Optional common frame**: the `FRAME_*` parameters change only the common representation
after alignment. They do not change the position relative to the coils.
Changing only this frame should preserve the physical currents.

Examples: 90° = a quarter turn; −30° = clockwise rotation; an X mirror alone =
`ANGLE=0, MIRROR_X=True, MIRROR_Y=False`. Two mirrors are equivalent to a 180° rotation.
After a change, rerun the notebook from this cell to recompute the corresponding
positions, fields and currents.
The acquisition folder and sequence are selected at the top of the notebook with
`INPUT_DIR` and `GRE_EPI_SEQUENCE_NAME`. The current selection is acquisition 029_(06),
GRE-EPI 128×128. The name is explicit even when several `.seq` files are present.
The raw scan must match this sequence; changing the name does not convert
a 64×64 acquisition into 128×128. After changing it, rerun from the imports.

**Multi-echo mapping**: `B0_SEQUENCE_NAME` selects the mapping `.seq` file,
and `B0_SCAN_PATH` selects its Twix acquisition. The echo count, TE, TR, dwell time,
FOV and matrix are checked. `B0_FIELD_ECHO_COUNT=6` uses the first six echoes
(up to 11.223 ms) to calculate B0; all 32 remain available for fitting R2*.

**Noise and support**: inspect the corners defined by `BACKGROUND_CORNER_M` and
the displayed masks. `NOISE_MASK=None` uses these corners; a native boolean mask
can also be supplied explicitly. They must contain no object signal. The SNR threshold
is applied after subtracting RSS noise power. Calibration is required by default;
a contaminated or missing background produces an explicit error rather than an invented SNR.

**CPU/GPU execution**: `GPU_PROFILE="auto"` selects the available GPU, and
`GPU_REQUIRED=True` requires actual CUDA execution. The check at the start of the
notebook displays the interpreter, PyTorch, CUDA and device; failure stops execution
before data preparation. The model and step 6 also enforce this requirement,
including when an old CPU model remains in memory.
For an intentional CPU run, select `GPU_REQUIRED=False` and `GPU_PROFILE="cpu"`.
Profiles do not change the installed PyTorch distribution: install a compatible
version in the notebook's Python environment, then restart the kernel; see
`../docs/SERVER_SETUP.md`. Changing these parameters alone does not move an existing
model to another device: rerun from the beginning.

For a shared distribution supporting the RTX PRO 6000 Blackwell, GTX 1660 Ti
and V100, install once in the intended kernel:
`%pip install --upgrade torch==2.8.0+cu128 --index-url https://download.pytorch.org/whl/cu128`,
then restart the kernel. This Linux distribution covers `sm_120`, `sm_75`
and `sm_70`; it also requires an NVIDIA driver compatible with CUDA 12.8.
A V100 can retain its CUDA 11.8 environment on its machine if necessary.
The CUDA 12.8 build of PyTorch 2.7.1 excluded `sm_70`; version 2.8 restores it.

**Legacy comparison**: with `ECHO_SIGNAL_MODEL="legacy"`, `RATE_MODE="guarded"` applies
reliable local fits at the EPI TE. `"pooled"` selects the conservative global rate;
`"none"` disables decay. The current criteria require sufficient SNR at every echo.
Voxels with weak late echoes can therefore be rejected; their fallback behavior is displayed.
The fitted rate and amplitude are effective quantities, not a proton-density map.

**Complex signal from the 32 echoes**: `ECHO_SIGNAL_MODEL="auto"` enables the new fit
for measured data; `"complex"` requests it explicitly, and `"legacy"` retains
the historical comparison. `RATE_MODE` applies to the legacy mode; the new mode uses
`effective_rate_mode="complex"`. Early-echo B0 initializes the fit; all 32 echoes
can then refine its central frequency. In-plane slopes remain those fixed before
fitting to avoid counting dephasing twice.

The fit compares three model families on the last `COMPLEX_VALIDATION_ECHOES` echoes,
without refitting amplitudes on those validation echoes. The selected model then uses
all echoes. Insufficient quality and parameter bounds remain flagged, without removing
signal voids or restoring zero decay. These checks are internal to the acquisition:
EPI times beyond the last echo still require extrapolation.

**Independent acquisition choices**: `GRE_EPI_SCAN_PATH`, `REFERENCE_SCAN_PATH`
and `B0_SCAN_PATH` each accept a name relative to `INPUT_DIR` or an absolute path.
Setting `REFERENCE_SCAN_PATH` to the new 64×64 or 128×128 `.dat` file is sufficient;
no particular suffix is required. With `None`, the legacy reference is found through
a unique `*reference*.dat` file. An ambiguous selection produces an error.
All three acquisitions retain their native physical grids and must cover the same slice;
a 64×64 reference is never treated as a 128×128 measurement.
The `*_SEQUENCE_NAME` values refer to files in the `sequences` subfolder or to absolute
paths. The reference `.seq` is optional; an explicitly selected `.seq` is checked
strictly against the selected `.dat` file.

**MRzero object constructed from GRE and the 32 echoes**:
`OBJECT_AMPLITUDE_SOURCE="joint"` uses GRE structure as a candidate signal envelope,
with a positive, spatially smooth correction for GRE→MEGRE contrast.
`OBJECT_CONTRAST_SIGMA_MM` sets its physical scale. This correction and the temporal
parameters are estimated without the held-out late echoes; those echoes select
the candidate's weight. Without a measured improvement, the model retains its
multi-echo amplitude, and the report states this. This calibration never rewrites
the GRE: it remains the independent measured target in step 6.
B0, decay and coherence come from the 32 echoes.

`"echo32"` explicitly retains the multi-echo envelope, `"auto"` tests the legacy
candidate with only a global gain, and `"reference"` forces that candidate.
A single GRE scan measures neither PD nor T1. The smooth correction represents
effective contrast between acquisitions, without separately identifying T1, B1
or receive sensitivity. Interpolating GRE64 onto the EPI128 grid does not create
detail acquired at 128².

**Image optimization target**: the selected measured GRE is used regardless of its matrix,
after interpolation into the common physical coordinates.
`OPTIMIZATION_REFERENCE_THRESHOLD=0` compares the full common coverage, including dark
pixels. A positive value defines a fixed GRE ROI. The simulation→EPI without ACDC and
EPI without ACDC→GRE gains are estimated once, then frozen. No gain or registration
is refitted at each iteration. The zero-B0 simulation remains a diagnostic;
it does not contribute to this target.

Calibration is shared through `PROJECT_ROOT/data/ACDC_3T.npz` using
`CALIBRATION_PATH`. CSV files are saved in `INPUT_DIR/waveform`.
`EXPORT_FIELD_SHIM_WAVEFORMS` enables static and dynamic exports independently
of image optimization.

**3D voxel**: the model is fixed to `finite_voxel` and `SUBSPINS_Z=1`.
The x/y slopes follow the native pixel axes; z denotes the slice normal.
Amplitude is not multiplied by thickness a second time.
All eight volume corners are checked for coverage and field limits.


In [ ]:
# Physical alignment of imaging data in the ACDC coil frame.
ALIGNMENT_ANGLE_DEG=90
ALIGNMENT_MIRROR_X=False
ALIGNMENT_MIRROR_Y=False

# Change the common representation without moving the object relative to the coils.
FRAME_ANGLE_DEG=90.0
FRAME_MIRROR_X=False
FRAME_MIRROR_Y=False

# Object and relaxation estimated from the 32 echoes.
OBJECT_THRESHOLD_FRACTION=0.04
BACKGROUND_CORNER_M=0.085
NOISE_MASK=None                 # Optional 128² boolean mask explicitly containing no signal.
OBJECT_MIN_SNR=5.0
OBJECT_MIN_COMPONENT_VOXELS=4
REQUIRE_NOISE_CALIBRATION=True  # False allows a no-relaxation fallback when noise calibration is unavailable.
B0_FIELD_ECHO_COUNT=6           # B0 initialization; the complex fit then uses all 32 echoes.
ECHO_SIGNAL_MODEL="auto"        # "auto": complex model for measured data; "legacy": old RSS fit.
COMPLEX_FIT_BATCH_SIZE=256
COMPLEX_FIT_ITERATIONS=60
COMPLEX_VALIDATION_ECHOES=6      # Hold out late echoes for model selection, then refit all 32.
COMPLEX_MAX_WIDTH_HZ=120.        # Bound fixed before fitting, without tuning on EPI.
OBJECT_AMPLITUDE_SOURCE="joint"  # GRE + 32 echoes, smooth contrast and selection on held-out echoes.
OBJECT_CONTRAST_SIGMA_MM=10.0    # Contrast scale between acquisitions, fixed before fitting.
REFERENCE_PRIOR_MIN_VALIDATION_IMPROVEMENT=0.02  # Minimum improvement on held-out echoes.
SIMULATION_MODEL="finite_voxel"  # This generation integrates a 3D voxel.
B0_WEIGHTING="auto"             # "auto", "uniform" or "echo_power".
BASELINE_CALIBRATION="auto"   # "auto" (measured data only), "off" or "on".
CALIBRATION_BOUNDS={"frequency_offset_Hz":30.,"readout_shift_mm":2.,"odd_even_phase_rad":.15}
B0_MODE="regularized"          # "regularized" or "raw".
B0_SIGMA_PIXELS=1.0             # Fixed native scale, without tuning on the EPI scan.
RATE_MODE="guarded"             # "pooled", "guarded" or "none".
SUBSPINS_Z=1                   # One full-thickness voxel, without z subdivisions.

GPU_PROFILE="auto"             # Detect GTX 1660 Ti, V100 and Blackwell without changing the notebook.
GPU_REQUIRED=True              # Stop if CUDA is unavailable. False allows CPU execution.
GPU_PRECISION="auto"
GPU_MEMORY_FRACTION=0.60         # Favor MRzero caching on the GPU; adapt blocks to available memory.
SAMPLE_CHUNK=None
RECONSTRUCTION_CHUNK=256

MAX_CURRENT_A=3.0
DISABLED_CHANNELS=[9]  # Disabled physical outputs (1..32); e.g. [7] or [7, 12].
TOTAL_CURRENT_LIMIT_A=30.0
FIELD_LIMIT_HZ=1000.0
CURRENT_RESPONSE={"enabled":True,"tau_s":133e-6/np.log(100.)}
HARDWARE={"min_update_interval_s":8e-6,"dac_resolution_A":0.0006,
          "max_slew_A_per_s":None,"avg_abs_current_A":None,
          "coil_R_ohm":None,"coil_L_H":None,"max_voltage_V":None,"max_coil_power_W":None}

IMAGE_INITIALIZATION="static"  # "static" or "zero".
# Physical guidance for severe EPI distortion/aliasing.
FIELD_GUIDANCE_WEIGHT=0.05      # Initial value to compare; 0 restores image-only loss.
FIELD_GUIDANCE_SCALE_HZ=100.0   # Fixed B0-residual normalization, rather than a field limit.
MAX_ITERATIONS=3000
LEARNING_RATE_A=0.01
PATIENCE=100
TARGET_MSE=None
OPTIMIZATION_REFERENCE_THRESHOLD=0.0  # 0: common GRE/EPI coverage, including dark pixels.
# >0: ROI above this fraction of the GRE maximum; no current-dependent mask.
LIVE_UPDATE_EVERY=20           # Reduce figure/HTML CPU overhead during GPU computation.
RUN_OPTIMIZATION=True

# Local export only; the controller is operated separately.
EXPORT_WAVEFORM=True         # Image-optimization waveform.
EXPORT_FIELD_SHIM_WAVEFORMS=True  # Static and dynamic, even without image optimization.
FIELD_SHIM_RASTER_S=10e-6
FIELD_SHIM_MAX_ITERATIONS=80
CSV_TIME_ORIGIN="first_imaging_trigger"  # Or "rf".
TRIGGER_CHANNEL="ext1"

# Current measurement: one time column, reset to zero at every trigger.
EXPORT_ADC_WAVEFORM=True
WAVEFORM_ADC_INTERVAL_S=0.000050
WAVEFORM_ADC_DURATION_S=None  # Total duration from the first trigger; None = end of the last readout.
WAVEFORM_PARAMETERS={"channel_map":list(range(1,33)),"channel_polarity":[-1]*32,
    "max_current_A":MAX_CURRENT_A,"total_current_limit_A":TOTAL_CURRENT_LIMIT_A,
    "min_dac_interval_s":8e-6,"disabled_channels":DISABLED_CHANNELS}
# One configuration for model constraints and all exports.
WAVEFORM_PARAMETERS=validate_waveform_settings(WAVEFORM_PARAMETERS,len(WAVEFORM_PARAMETERS["channel_map"]))
CHANNEL_MAX_CURRENT_A=np.asarray(WAVEFORM_PARAMETERS["max_current_A"],dtype=float)

CALIBRATION_PARAMETERS={"units":"Hz/A","registration_confirmed":True,
    "patient_to_source_m":np.eye(4).tolist(),
    "source_sha256":"bace6a584e780d80af8f2ca7c12957bd5b9dc7462be0968f949cc9a99243a06c",
    "registration_note":"Bay5 placement previously confirmed; imaging orientation configured above."}
# Final comparison: None uses all *scan.dat files in the results folder.
# To select: ["corrected_scan.dat"] (example filename).
CORRECTED_SCAN_NAMES=None
NORMALIZE_COMPARISON=True
COMPARISON_REFERENCE_THRESHOLD=0.1  # GRE ROI; 0 uses the entire common coverage.
WAVEFORM_PREVIEW_CHANNEL=None      # Physical output (1..32), or the largest change.

In [ ]:
def table(rows,headings=("Parameter","Value")):
    display(Markdown("| "+" | ".join(headings)+" |\n| "+" | ".join(["---"]*len(headings))+" |\n"+
        "\n".join("| "+" | ".join(str(v) for v in row)+" |" for row in rows)))

def show(fig):
    plt.show();plt.close(fig)

def image_physical(ax,image,X,Y,title,cmap="gray",**options):
    artist=ax.pcolormesh(X*1000,Y*1000,np.asarray(image),shading="auto",cmap=cmap,**options)
    ax.set(title=title,xlabel="X [mm]",ylabel="Y [mm]",aspect="equal")
    return artist

def json_safe(value):
    if isinstance(value,dict):return {str(k):json_safe(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):return [json_safe(v) for v in value]
    if isinstance(value,np.ndarray):return json_safe(value.tolist())
    if isinstance(value,np.generic):return json_safe(value.item())
    if isinstance(value,Path):return str(value)
    if isinstance(value,float) and not np.isfinite(value):return None
    return value

angles={"alignment_angle_deg":ALIGNMENT_ANGLE_DEG,"alignment_mirror_x":ALIGNMENT_MIRROR_X,
    "alignment_mirror_y":ALIGNMENT_MIRROR_Y,"frame_angle_deg":FRAME_ANGLE_DEG,
    "frame_mirror_x":FRAME_MIRROR_X,"frame_mirror_y":FRAME_MIRROR_Y}
Q=inplane_matrix(ALIGNMENT_ANGLE_DEG,ALIGNMENT_MIRROR_X,ALIGNMENT_MIRROR_Y)
P=inplane_matrix(FRAME_ANGLE_DEG,FRAME_MIRROR_X,FRAME_MIRROR_Y)
O=P@Q
table(list(angles.items()))
table([("Disabled physical outputs",WAVEFORM_PARAMETERS["disabled_channels"]),
       ("Number of active channels",int(np.count_nonzero(CHANNEL_MAX_CURRENT_A)))])
print("Data→coil matrix Q:\n",Q,"\nCompute-frame matrix O=P@Q:\n",O)
print("Output folder:",OUTPUT_DIR)

# Check the GPU before reading and preparing acquisitions.
print("Kernel Python:",sys.executable)
print("PyTorch / CUDA runtime:",torch.__version__,"/",torch.version.cuda)
execution_device,execution_dtype,execution_report=select_device(
    GPU_PROFILE,GPU_PRECISION,require_cuda=GPU_REQUIRED)
table([("Compute device",str(execution_device)),
       ("GPU",execution_report["gpu_name"]),
       ("Precision",execution_report["precision"]),
       ("GPU required",GPU_REQUIRED)])


## 2. Check the sequence and prepare the 32 echoes (B0 map)

The check below reads the actual Pulseq events before the Twix data.
It displays the ADC sample count, TEs, TR and EPI timings. In the supplied file,
the auxiliary `EchoSamples` definition states 64, whereas the ADC events and `Matrix`
correctly indicate 128: the reader checks the actual events.
Small kx offsets between echoes are retained in the complex reconstruction.

The subsequent read checks that the `.dat` matches this sequence.
Noise is measured **before** the object mask, using only the explicitly selected
corners or mask. No region is automatically replaced by the complement of the object.
The figures allow inspection of noise, support and B0 coverage.
B0 phase uses the early echoes; the relaxation fit retains all 32.

The selected GRE reference is displayed immediately after loading, in its native
physical coordinates, with its filename, matrix, TE and TR, before fitting the 32 echoes.


In [ ]:
import warnings
import importlib
import read_acdc_b0 as b0_reader
# The kernel may have loaded the old reader from 05.
b0_reader=importlib.reload(b0_reader)
read_b0_sequence=b0_reader.read_b0_sequence

def selected_sequence_path(name):
    if name is None:
        return None
    path=Path(name)
    return path if path.is_absolute() else INPUT_DIR/"sequences"/path

selected_epi_sequence=selected_sequence_path(GRE_EPI_SEQUENCE_NAME)
selected_b0_sequence=selected_sequence_path(B0_SEQUENCE_NAME)
selected_reference_sequence=selected_sequence_path(REFERENCE_SEQUENCE_NAME)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=r"^File version 1\.5\.1 is higher than installed package version 1\.5\.0\. This may cause parsing errors\.$",
                            category=UserWarning, module=r"^pypulseq\.Sequence\.read_seq$")
    b0_sequence_report=read_b0_sequence(selected_b0_sequence)
    epi_preview,_=read_sequence(selected_epi_sequence)
if b0_sequence_report["echo_count"] != 32 or tuple(b0_sequence_report["image_shape"]) != (128,128):
    raise ValueError("The notebook expects a 32-echo mapping acquisition with a 128×128 matrix.")
if not 2 <= B0_FIELD_ECHO_COUNT <= b0_sequence_report["echo_count"]:
    raise ValueError("B0_FIELD_ECHO_COUNT must select between 2 and 32 echoes.")
expected_echo_times=np.asarray(b0_sequence_report["TE_s"])
epi_relative_times=np.asarray(epi_preview["time_s"])-np.asarray(epi_preview["rf_time_s"])
sequence_assessment={"B0_sequence":B0_SEQUENCE_NAME,"B0_expected_echo_count":32,
    "B0_field_echo_count":B0_FIELD_ECHO_COUNT,"echo_times_s":expected_echo_times,
    "target_TE_s":float(epi_preview["TE_s"]),
    "EPI_ADC_range_s":[float(epi_relative_times.min()),float(epi_relative_times.max())],
    "extrapolation_to_TE_s":max(0.,float(epi_preview["TE_s"]-expected_echo_times[-1])),
    "extrapolation_to_last_ADC_s":max(0.,float(epi_relative_times.max()-expected_echo_times[-1]))}
table([("Mapping sequence",B0_SEQUENCE_NAME),
       ("Echoes / ADC matrix",f"{b0_sequence_report['echo_count']} / {b0_sequence_report['image_shape']}"),
       ("First / last TE [ms]",expected_echo_times[[0,-1]]*1000),
       ("Mapping TR [ms]",b0_sequence_report["TR_s"]*1000),
       ("Mapping duration [s]",b0_sequence_report["sequence_duration_s"]),
       ("EPI TE [ms]",epi_preview["TE_s"]*1000),
       ("First / last ADC after RF [ms]",np.array(sequence_assessment["EPI_ADC_range_s"])*1000),
       ("Extrapolation from last echo → EPI TE [ms]",sequence_assessment["extrapolation_to_TE_s"]*1000),
       ("Extrapolation from last echo → last ADC [ms]",sequence_assessment["extrapolation_to_last_ADC_s"]*1000)])
print("Mapping Twix:",B0_SCAN_PATH if B0_SCAN_PATH is not None else "detect a unique *b0_map.dat; 32 echoes required")
print("Reference Twix:",REFERENCE_SCAN_PATH if REFERENCE_SCAN_PATH is not None else "unique *reference*.dat")
print("Reference sequence:",selected_reference_sequence or "automatically select a compatible sequence, otherwise nominal Twix Cartesian coordinates")
print("EPI Twix:",GRE_EPI_SCAN_PATH if GRE_EPI_SCAN_PATH is not None else "unique *scan.dat excluding reference/B0 files")


In [ ]:
# Reload the reader after a code change in the same kernel.
import importlib
import twin_inputs
twin_inputs=importlib.reload(twin_inputs)
read_twin_inputs=twin_inputs.read_twin_inputs
prepare_echo_object=twin_inputs.prepare_echo_object

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=r"^File version 1\.5\.1 is higher than installed package version 1\.5\.0\. This may cause parsing errors\.$",
                            category=UserWarning, module=r"^pypulseq\.Sequence\.read_seq$")
    inputs=read_twin_inputs(INPUT_DIR,
        scan_sequence_path=selected_epi_sequence,scan_path=GRE_EPI_SCAN_PATH,
        reference_sequence_path=selected_reference_sequence,reference_scan_path=REFERENCE_SCAN_PATH,
        b0_sequence_path=selected_b0_sequence,
        b0_scan_path=B0_SCAN_PATH,b0_echo_count=B0_FIELD_ECHO_COUNT,
        reconstruction_device=execution_device,reconstruction_dtype=execution_dtype)
# Display the selected reference before fitting the 32 echoes.
print("GRE reference:",inputs["paths"]["reference"].resolve())
print("GRE sequence actually used:",inputs["paths"].get("reference_sequence","none — nominal Twix Cartesian trajectory"))
table([("Native reference matrix",np.asarray(inputs["gre"]["magnitude"]).shape),
       ("Reference TE [ms]",float(inputs["gre"].get("TE_s",np.nan))*1000),
       ("Reference TR [ms]",float(inputs["gre"].get("TR_s",np.nan))*1000),
       ("Reference validation",inputs.get("reference_sequence_validation",{})),
       ("Slice checks",inputs.get("geometry_checks",[]))])
reference_gre=inputs["gre"]
reference_magnitude=np.asarray(reference_gre["magnitude"])
reference_rows,reference_columns=reference_magnitude.shape
fig,ax=plt.subplots(figsize=(8,6),layout="constrained")
reference_artist=image_physical(ax,reference_magnitude,
    reference_gre["X_m"],reference_gre["Y_m"],
    f"Measured GRE reference — {reference_rows}×{reference_columns}\n"
    f"{inputs['paths']['reference'].name}",vmin=0)
fig.colorbar(reference_artist,ax=ax,label="RSS magnitude [a.u.]")
show(fig)

if len(inputs["echo_times_s"]) != 32:
    raise ValueError("The selected Twix file does not contain the expected 32 echoes.")
fit=prepare_echo_object(inputs,OBJECT_THRESHOLD_FRACTION,BACKGROUND_CORNER_M,
    allow_uncalibrated_noise=not REQUIRE_NOISE_CALIBRATION,noise_mask=NOISE_MASK,
    support_min_snr=OBJECT_MIN_SNR,min_component_voxels=OBJECT_MIN_COMPONENT_VOXELS)
# Support and noise remain independent of the new temporal model.
legacy_echo_fit=fit
if ECHO_SIGNAL_MODEL not in {"auto","complex","legacy"}:
    raise ValueError("ECHO_SIGNAL_MODEL must be auto, complex or legacy.")
input_is_synthetic=(inputs.get("manifest") or {}).get("format") in {
    "mrzero_synthetic_acquisition","mrzero_synthetic_twix"}
complex_echo_enabled=(ECHO_SIGNAL_MODEL=="complex" or
                      (ECHO_SIGNAL_MODEL=="auto" and not input_is_synthetic))
effective_rate_mode="complex" if complex_echo_enabled else RATE_MODE
complex_echo_diagnostics=None
if complex_echo_enabled:
    echo_weighting=("echo_power" if B0_WEIGHTING=="auto" else B0_WEIGHTING)
    fit=prepare_complex_echo_model(inputs,legacy_echo_fit,
        device=str(execution_device),require_cuda=GPU_REQUIRED,
        b0_sigma_pixels=B0_SIGMA_PIXELS,b0_weighting=echo_weighting,
        batch_size=COMPLEX_FIT_BATCH_SIZE,max_iterations=COMPLEX_FIT_ITERATIONS,
        max_width_Hz=COMPLEX_MAX_WIDTH_HZ,validation_echo_count=COMPLEX_VALIDATION_ECHOES)
    fit=apply_reference_amplitude_prior(inputs,fit,
        source=OBJECT_AMPLITUDE_SOURCE,contrast_sigma_mm=OBJECT_CONTRAST_SIGMA_MM,
        device=str(execution_device),require_cuda=GPU_REQUIRED,
        validation_echo_count=COMPLEX_VALIDATION_ECHOES,
        min_relative_validation_improvement=REFERENCE_PRIOR_MIN_VALIDATION_IMPROVEMENT,
        fit_options={"batch_size":COMPLEX_FIT_BATCH_SIZE,"max_iterations":COMPLEX_FIT_ITERATIONS,
                     "max_width_Hz":COMPLEX_MAX_WIDTH_HZ})
    table(list(fit["report"]["reference_amplitude_prior"].items()),("Reference contribution to amplitude","Value"))
    complex_echo_diagnostics=multiecho_comparison(inputs,fit)
    table(list(fit["report"]["multi_echo"].items()),("Complex 32-echo fit","Value"))
    table([(row["region"],row["pixels"],row["legacy_RSS_RMSE_over_first_echo"],
            row["complex_RSS_RMSE_over_first_echo"]) for row in complex_echo_diagnostics["rows"]],
          ("MEGRE region","Pixels","Legacy-model error / echo 1","Complex-model error / echo 1"))
print("Temporal model used by MRzero:",effective_rate_mode)
noise_info=fit["report"]["noise"]
support_info=fit["report"]["support"]
raw=inputs["b0_raw"]
acquisition_native=inputs["acquisition"]
pooled=fit["pooled"]
print("Mapping Twix:",inputs["paths"]["b0"].resolve())
print("Mapping sequence:",inputs["paths"]["b0_sequence"].resolve())
print("EPI scan without ACDC:",inputs["paths"]["scan"].resolve())
print(f"Noise: {noise_info['available_sample_count']} finite complex vectors / "
      f"{noise_info['minimum_sample_count']} required; calibrated = {noise_info['calibrated']}.")
if not noise_info["calibrated"]:
    print("Fallback: first echo, without additional R2* decay. Reason:",noise_info["fallback_reason"])
table([("Echoes read for relaxation",len(inputs["echo_times_s"])),
       ("Early echoes used for B0",inputs["field_echo_count"]),
       ("Mapping TE [ms]",inputs["echo_times_s"]*1000),
       ("EPI TE [ms]",acquisition_native["TE_s"]*1000),
       ("Independent noise pixels",int(fit["noise_mask"].sum())),
       ("Object-support pixels",int(fit["object_support"].sum())),
       ("Simulatable pixels with B0 coverage",int(fit["simulation_support"].sum())),
       ("Diagnostic global R2* rate [s⁻¹]",pooled.get("R2star_sInv")),
       ("Global-rate uncertainty [s⁻¹]",pooled.get("R2star_std_sInv")),
       ("Global rate accepted at EPI TE",pooled.get("target_reliable_mask",False)),
       ("Legacy RSS fit: fits accepted at EPI TE",int(fit["target_reliable_mask"].sum()))])
table(list(support_info.items()),("Support diagnostic","Value"))

raw_first_rss=np.sqrt(np.sum(np.abs(raw["coil_images"][0])**2,axis=0))
accepted_rates=(np.where(fit["multi_echo"]["support_mask"],fit["multi_echo"]["residual_R2_sInv"],np.nan)
                if complex_echo_enabled else np.where(fit["target_reliable_mask"],fit["R2star_sInv"],np.nan))
fig,axes=plt.subplots(2,3,figsize=(15,9),layout="constrained")
images=[raw_first_rss,fit["support_debiased_amplitude"],fit["object_support"],
        fit["noise_mask"],inputs["field"]["b0Hz"],accepted_rates]
amplitude_title=("Amplitude after RSS noise removal" if support_info["amplitude_debiased"]
                 else "Raw RSS: noise not calibrated")
titles=["Raw first-echo RSS",amplitude_title,"SNR-based object support",
        "Independent noise mask","B0: early echoes [Hz]",("Complex residual decay [s⁻¹]" if complex_echo_enabled else "Local R2* accepted at EPI TE [s⁻¹]")]
for ax,image,title in zip(axes.ravel(),images,titles):
    im=image_physical(ax,image,raw["X_m"],raw["Y_m"],title,
                      cmap="RdBu_r" if "[Hz]" in title else "gray")
    fig.colorbar(im,ax=ax,shrink=.8)
show(fig)

### Measure the currents: `waveform_adc.csv`

`WAVEFORM_ADC_INTERVAL_S=0.000050` sets the interval between measurements (50 µs).
The CSV contains one numeric column, without a header: `0`, `0.000050`,
`0.000100`, etc. The counter restarts at zero at each `ext1` trigger verified
from the sequence, including sequences with one trigger per line.

By default, the window runs from the first trigger to the end of the last imaging
readout. `WAVEFORM_ADC_DURATION_S` sets a different total duration from the first
trigger, in seconds. The next trigger or the end of the window is excluded from
the previous segment. The associated JSON retains absolute trigger times and
sample-block indices.

The file is saved in `INPUT_DIR/waveform`, independently of image optimization
and field shims. Rerunning reuses an identical file or adds a number if its
parameters change.


In [ ]:
adc_waveform_export=None
if EXPORT_ADC_WAVEFORM:
    from adc_waveform_export import export_adc_waveform
    adc_waveform_export=export_adc_waveform(
        inputs["paths"]["scan_sequence"],inputs["timing"],WAVEFORM_OUTPUT_DIR,
        interval_s=WAVEFORM_ADC_INTERVAL_S,trigger_channel=TRIGGER_CHANNEL,
        duration_s=WAVEFORM_ADC_DURATION_S)
    print("ADC CSV:",adc_waveform_export["csv_file"])
    print("ADC interval [s]:",adc_waveform_export["interval_s"],
          "| samples:",adc_waveform_export["samples"],
          "| triggers:",adc_waveform_export["trigger_count"])
else:
    print("ADC export disabled.")


In [ ]:
# Diagnostic of the old global RSS fit; the spatial complex model is checked below.
times=np.asarray(inputs["echo_times_s"])
mean_echoes=np.asarray(fit["pooled"].get("mean_RSS_echoes",[]))
fig,ax=plt.subplots(figsize=(10,4),layout="constrained")
if len(mean_echoes)==len(times):
    ax.plot(times*1000,mean_echoes,"o",label="Mean debiased amplitudes in the global ROI")
    rate=fit["pooled"].get("R2star_sInv",np.nan)
    anchor_amplitude=fit["pooled"].get("fitted_amplitude_at_reference",np.nan)
    if np.isfinite(rate) and np.isfinite(anchor_amplitude):
        prediction_time=np.linspace(times[0],max(times[-1],sequence_assessment["EPI_ADC_range_s"][1]),300)
        prediction=anchor_amplitude*np.exp(-rate*(prediction_time-times[0]))
        ax.plot(prediction_time*1000,prediction,"--",label="Global fit (acceptance shown in the table)")
ax.axvspan(times[0]*1000,times[-1]*1000,color="tab:green",alpha=.1,label="Measured multi-echo range")
ax.axvline(acquisition_native["TE_s"]*1000,color="tab:red",label="EPI TE")
ax.axvline(sequence_assessment["EPI_ADC_range_s"][1]*1000,color="tab:gray",linestyle=":",label="Last EPI ADC")
ax.set(xlabel="Time since RF [ms]",ylabel="Effective amplitude",title="Global RSS diagnostic: temporal coverage")
ax.legend(fontsize=8);show(fig)
if not complex_echo_enabled and not fit["target_reliable_mask"].any():
    print("No local fit is accepted at EPI TE: check late-echo SNR and diagnostics before interpreting the correction.")

### 2b — Check signal voids and late-echo prediction

Each column compares the same measured MEGRE image, the legacy prediction with
fallbacks, the new complex prediction, and its error, using a shared scale.
The signal-void mask is descriptive: pixels bright at the first echo that fall below
25% at echo 15. It changes neither support nor fitting loss.
The comparison after refitting all 32 echoes is separate from the held-out echo errors
recorded in the model-selection report.
All four rows use the same measured B0 support; uncovered pixels are masked.


In [ ]:
if complex_echo_enabled:
    ce=fit["multi_echo"]
    echo_indices=[0,7,14,19,25,31]
    measured=fit["debiased_RSS"]
    predicted=ce["predicted_RSS"]
    previous=complex_echo_diagnostics["legacy_predicted_RSS"]
    vmax=float(np.nanpercentile(measured[0][ce["support_mask"]],99))
    fig,axes=plt.subplots(4,len(echo_indices),figsize=(21,13),layout="constrained")
    for column,index in enumerate(echo_indices):
        for row,(image,label) in enumerate([(measured[index],"Measured"),
            (previous[index],"Legacy model"),(predicted[index],"Complex fit"),
            (predicted[index]-measured[index],"Fit error")]):
            signed=row==3
            axes[row,column].set_facecolor("black")
            image_physical(axes[row,column],np.where(ce["support_mask"],image,np.nan),raw["X_m"],raw["Y_m"],
                f"{label}: echo {index+1} / {inputs['echo_times_s'][index]*1000:.1f} ms",
                cmap="RdBu_r" if signed else "gray",vmin=-vmax if signed else 0,vmax=vmax)
    show(fig)
    regions=[("Measured support",ce["support_mask"]),
             ("Signal voids at echo 15",complex_echo_diagnostics["hole_mask"])]
    fig,axes=plt.subplots(1,2,figsize=(13,4),layout="constrained")
    for ax,(label,region) in zip(axes,regions):
        if not region.any():
            ax.set_title(label+": no pixels");continue
        scale=np.maximum(fit["first_echo_amplitude"][region],np.finfo(float).tiny)
        for data,name,style in [(measured,"Measured","o"),(previous,"Legacy model","--"),
                                (predicted,"Complex fit","-")]:
            ax.plot(inputs["echo_times_s"]*1000,np.median(data[:,region]/scale[None],axis=1),
                    style,label=name)
        ax.axvspan(inputs["echo_times_s"][-COMPLEX_VALIDATION_ECHOES]*1000,
                   inputs["echo_times_s"][-1]*1000,color="tab:green",alpha=.1,
                   label="Held-out echoes before refitting")
        ax.set(title=label,xlabel="TE [ms]",ylabel="Amplitude / first echo")
        ax.legend(fontsize=8)
    show(fig)

## 3. Orient the object and prepare one 3D voxel per pixel

Amplitude and B0 remain attached to the native grid. Alignment Q places the object
in the coil maps; P transforms the common Fourier frame.
ACDC maps are sampled at the center, the endpoints of all three axes and all eight
corners of each voxel. A voxel without full coverage is excluded, without renormalizing
the remaining amplitude. Gradients represent frequency changes across the **full**
width of each voxel axis.


In [ ]:
# Synthetic acquisitions were generated with point spins.
# Measured data use the voxel-integrating model by default.
manifest_format=(inputs.get("manifest") or {}).get("format")
is_point_synthetic=manifest_format in {"mrzero_synthetic_acquisition","mrzero_synthetic_twix"}
if SIMULATION_MODEL!="finite_voxel" or SUBSPINS_Z!=1:
    raise ValueError("Generations 08/09 use finite_voxel and SUBSPINS_Z=1.")
if B0_WEIGHTING not in {"auto","uniform","echo_power"}:
    raise ValueError("B0_WEIGHTING must be auto, uniform or echo_power.")
effective_simulation_model="finite_voxel"
effective_b0_weighting=(("uniform" if is_point_synthetic else "echo_power")
                        if B0_WEIGHTING=="auto" else B0_WEIGHTING)
prepared=prepare_voxel_inputs(inputs,fit,CALIBRATION_PATH,CALIBRATION_PARAMETERS,
    **angles,rate_mode=effective_rate_mode,b0_mode=B0_MODE,
    b0_sigma_pixels=B0_SIGMA_PIXELS,current_response=CURRENT_RESPONSE,
    b0_weighting=effective_b0_weighting)
experiment=prepared["expanded_experiment"]
calibration=prepared["expanded_calibration"]
preparation_report=prepared["report"]
assert np.allclose(prepared["combined_frame_matrix"],O,rtol=0,atol=1e-12)
assert preparation_report["all_children_in_field_limits"]
table([(key,preparation_report[key]) for key in ["requested_parent_count","baseline_B0_parent_count",
    "retained_parent_count","subspin_count","excluded_missing_B0_amplitude_fraction",
    "excluded_ACDC_amplitude_fraction_of_baseline","amplitude_area_ratio","slice_thickness_mm"]])
print("Mirror / rotation: no image-array interpolation.")
if effective_simulation_model=="finite_voxel":
    table([(key,preparation_report[key]) for key in ["inplane_voxel_enabled",
        "excluded_missing_B0_gradient_parent_count","excluded_missing_ACDC_gradient_or_corner_parent_count",
        "inplane_voxel_field_limit_corners_per_child","ACDC_corner_linearization_max_abs_error_Hz_per_A"]],
        ("3D voxel integration","Value"))

fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
image_physical(axes[0],prepared["native_amplitude"],prepared["physical_X_m"],prepared["physical_Y_m"],
               "Object in the coil frame")
image_physical(axes[1],prepared["native_B0_Hz"],prepared["physical_X_m"],prepared["physical_Y_m"],
               "B0 in the coil frame [Hz]",cmap="RdBu_r")
image_physical(axes[2],prepared["parent_valid_mask"],prepared["physical_X_m"],prepared["physical_Y_m"],
               "Full-voxel coverage (eight corners)")
for column,color,label in [(0,"tab:red","+X data"),(1,"tab:blue","+Y data")]:
    axes[0].arrow(0,0,25*Q[0,column],25*Q[1,column],color=color,width=.8,length_includes_head=True)
    axes[0].text(30*Q[0,column],30*Q[1,column],label,color=color,fontsize=8)
show(fig)
# Original snapshot for repeated construction and calibration.
uncalibrated_prepared=deepcopy(prepared)

## 4. First MRzero simulation and comparison with measured EPI

The effective signal is anchored at the first MEGRE TE. MRzero receives an amplitude
extrapolated to RF time and applies native relaxation. This amplitude includes measured
contrast and receive sensitivity: it is not quantitative PD.
The model retains the independent ideal excitations of 06; it does not simulate
the scan's T1 steady state. The following report makes these assumptions explicit.
The zero-B0 simulation is a diagnostic; measured GRE remains the optimization target.


In [ ]:
def reconstruction_factory(acquisition,device,dtype):
    return FrameReconstruction(acquisition,device,dtype,frame_matrix=O,
                               sample_chunk=RECONSTRUCTION_CHUNK)

# Restart from uncalibrated preparation even when rerunning only this cell.
prepared=deepcopy(uncalibrated_prepared)
experiment=prepared["expanded_experiment"]
calibration=prepared["expanded_calibration"]
preparation_report=prepared["report"]
if "model" in globals():
    import gc
    del model
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

model_class=NativeB0ACDCModel
model=model_class(experiment,calibration,max_current_A=CHANNEL_MAX_CURRENT_A,
    hardware=HARDWARE,reconstruction_factory=reconstruction_factory,
    profile=GPU_PROFILE,precision=GPU_PRECISION,memory_fraction=GPU_MEMORY_FRACTION,
    sample_chunk=SAMPLE_CHUNK,require_cuda=GPU_REQUIRED)
table([("Device / precision",f"{model.device} / {model.runtime_info['precision']}"),
       ("PyTorch / CUDA runtime",f"{torch.__version__} / {torch.version.cuda}"),
       ("Detected GPU",model.runtime_info["detected_gpu_name"])])
X_image,Y_image=model.reconstruction.X_m,model.reconstruction.Y_m
measured_coils=model.reconstruction.numpy_image(
    np.moveaxis(inputs["scan_diagnostics"]["corrected_kspace"],1,0))
measured_image=np.sqrt(np.sum(abs(measured_coils)**2,axis=0))
zero_currents=np.zeros((model.nc,model.nstates))
ideal_image=model.ideal_image
baseline_image,baseline_kdata=model.forward(zero_currents)
baseline_model_mse=float(np.sum((abs(baseline_image)-abs(ideal_image))**2)/np.sum(abs(ideal_image)**2))
baseline_measured_metrics=compare_magnitude(measured_image,baseline_image,X_m=X_image,Y_m=Y_image)
table([("Simulated spins",int(model.support.sum())),("MRzero cache storage",model.runtime_info["native_baseline_cache"]),
       ("Diagnostic MSE without ACDC / zero-B0 simulation",baseline_model_mse),
       ("Twin / actual EPI NRMSE [%]",baseline_measured_metrics["NRMSE_percent"])])
point_baseline_image=None
point_baseline_metrics=None
fidelity_comparison=None
gain=baseline_measured_metrics["gain"]
fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
for ax,image,title in zip(axes,[gain*np.abs(ideal_image),gain*np.abs(baseline_image),measured_image],
                         ["MRzero: zero B0","MRzero: measured B0","Actual EPI without ACDC"]):
    image_physical(ax,image,X_image,Y_image,title,vmin=0,vmax=measured_image.max()*1.1)
show(fig)
# Immutable snapshots: calibration never starts from its own result.
uncalibrated_baseline_image=baseline_image.copy()
uncalibrated_ideal_image=ideal_image.copy()
uncalibrated_baseline_kdata=baseline_kdata.copy()
uncalibrated_sample_chunk=int(model.sample_chunk)  # Retain the native partition when rebuilding.
uncalibrated_baseline_model_mse=baseline_model_mse
uncalibrated_baseline_measured_metrics=deepcopy(baseline_measured_metrics)

# Rebuilding invalidates the previous diagnostic.
baseline_signal_disagreement=None
baseline_signal_disagreement_arrays={}

### First-simulation parameters: MRzero / actual scan

A snapshot **before baseline calibration**, saved as JSON and NPZ.
The table also shows differences in RF/relaxation assumptions between the simulation
and the scan. Statistics include only the voxels actually simulated.


In [ ]:
first_simulation,first_simulation_rows=first_simulation_report(
    model,prepared,inputs,measured_image,baseline_image,baseline_kdata,GENERATION)
table(first_simulation_rows,("Parameter","First MRzero simulation","Actual scan"))
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
(OUTPUT_DIR/"first_simulation.json").write_text(
    json.dumps(first_simulation,indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
np.savez_compressed(OUTPUT_DIR/"first_simulation.npz",
    baseline_image=baseline_image,baseline_kdata=baseline_kdata,measured_image=measured_image,
    zero_b0_image=ideal_image,X_m=X_image,Y_m=Y_image,
    B0_Hz=experiment["field"]["b0Hz"],amplitude_at_reference=experiment["m0_proxy"],
    T2_s=experiment["relaxation"]["T2star_s"],
    voxel_B0_gradient_Hz_per_pixel=experiment["inplane_voxel"]["b0_gradient_Hz_per_pixel"])
print("First-simulation snapshot:",OUTPUT_DIR/"first_simulation.json")

### 4b — Calibrate small discrepancies in EPI without ACDC

`BASELINE_CALIBRATION="auto"` enables this step for measured acquisitions; synthetic
datasets remain unchanged. `"off"` disables it, and `"on"` requests it explicitly.
Bounds are fixed before fitting: global frequency offset ±30 Hz,
readout translation ±2 mm and constant residual odd/even phase ±0.15 rad.
Phase-encoding translation and rotation remain fixed because they are partly
confounded with frequency offset or coil alignment.

The object, relaxation and spatial B0 variations remain fixed. Separate spatial blocks
are used for fitting, selection and final testing; display gains are estimated only
on fitting pixels. This is **internal validation within a single acquisition** that
has already been used to develop the model, rather than an independent scan.
A small improvement may be insufficient to explain the remaining artifacts.

Application is rejected if selection does not improve, a bound is reached, parameters
are poorly identified, or the final test does not improve.
The selected parameters rebuild the native MRzero model: physical translation before
resampling ACDC maps, B0 offset, and residual ADC phase after navigator correction.
The zero-B0 diagnostic removes the frequency offset and retains the calibrated geometry
and hardware phase. This step changes neither currents nor measured data.

The preceding point/voxel figures describe the model **before calibration**.
Rerunning this cell starts from the original data, including after rejection or disabling.
After changing this step, rerun verification and optimization.


In [ ]:
# Rerun from the same uncalibrated samples without accumulating parameters.
import gc
import importlib
import baseline_calibration as baseline_calibration_tools
import baseline_metadata_audit as baseline_audit_tools
baseline_calibration_tools=importlib.reload(baseline_calibration_tools)
baseline_audit_tools=importlib.reload(baseline_audit_tools)
if BASELINE_CALIBRATION not in {"auto","off","on"}:
    raise ValueError("BASELINE_CALIBRATION must be auto, off or on.")
baseline_calibration_enabled=(BASELINE_CALIBRATION=="on" or
    (BASELINE_CALIBRATION=="auto" and not is_point_synthetic))
baseline_metadata_audit=baseline_audit_tools.audit_baseline_inputs(inputs)
baseline_calibration_result=None
baseline_calibration_masks={name:np.zeros(measured_image.shape,dtype=bool)
                            for name in ("fit_mask","validation_mask","test_mask")}
baseline_calibration_parameters={name:0. for name in baseline_calibration_tools.PARAMETER_NAMES}
if baseline_calibration_enabled:
    baseline_calibration_result=baseline_calibration_tools.fit_baseline_calibration(
        inputs["acquisition"],uncalibrated_baseline_kdata,measured_image,bounds=CALIBRATION_BOUNDS)
    baseline_calibration_report=deepcopy(baseline_calibration_result["report"])
    baseline_calibration_parameters=baseline_calibration_result["parameters"].copy()
    for name in baseline_calibration_masks:
        baseline_calibration_masks[name]=baseline_calibration_result[name].copy()
else:
    baseline_calibration_report={"accepted":False,"parameters":baseline_calibration_parameters.copy(),
        "reason":"Calibration disabled for this synthetic dataset." if is_point_synthetic
                 else "Calibration disabled by BASELINE_CALIBRATION=off.",
        "gain_applied_to_object":False,"independent_acquisition_validation":False}
baseline_calibration_report["requested_mode"]=BASELINE_CALIBRATION
baseline_calibration_report["source_sha256"]=dict(inputs["source_sha256"],
    calibration=uncalibrated_prepared["report"]["calibration_source_sha256"])

def rebuild_baseline_model(parameters):
    rebuilt=prepare_voxel_inputs(inputs,fit,CALIBRATION_PATH,CALIBRATION_PARAMETERS,
        **angles,rate_mode=effective_rate_mode,b0_mode=B0_MODE,
        b0_sigma_pixels=B0_SIGMA_PIXELS,current_response=CURRENT_RESPONSE,
        b0_weighting=effective_b0_weighting,
        baseline_calibration=parameters)
    rebuilt_model=model_class(rebuilt["expanded_experiment"],rebuilt["expanded_calibration"],
        max_current_A=CHANNEL_MAX_CURRENT_A,hardware=HARDWARE,reconstruction_factory=reconstruction_factory,
        profile=GPU_PROFILE,precision=GPU_PRECISION,memory_fraction=GPU_MEMORY_FRACTION,
        sample_chunk=uncalibrated_sample_chunk,require_cuda=GPU_REQUIRED)
    return rebuilt,rebuilt_model

# Release the old model's native cache before building the next one.
del model
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()
prepared,model=rebuild_baseline_model(baseline_calibration_parameters)
zero_currents=np.zeros((model.nc,model.nstates))
baseline_image,baseline_kdata=model.forward(zero_currents)
expected_calibrated_kdata=(uncalibrated_baseline_kdata*
    baseline_calibration_tools.phase_factor(inputs["acquisition"],baseline_calibration_parameters))
baseline_calibration_native_phase_parity=float(np.linalg.norm(baseline_kdata-expected_calibrated_kdata)
    /max(np.linalg.norm(expected_calibrated_kdata),np.finfo(float).tiny))
def fixed_object_rebuild_checks(candidate):
    checks={key:bool(np.array_equal(candidate[key],uncalibrated_prepared[key],equal_nan=True))
        for key in ("parent_flat_index","parent_valid_mask","weights","native_amplitude","native_rate_sInv")}
    checks["subspin_amplitude"]=bool(np.array_equal(candidate["expanded_experiment"]["m0_proxy"],
        uncalibrated_prepared["expanded_experiment"]["m0_proxy"],equal_nan=True))
    return checks
baseline_fixed_object_checks=fixed_object_rebuild_checks(prepared)
baseline_calibration_report["fixed_object_rebuild_checks"]=baseline_fixed_object_checks.copy()
if baseline_calibration_native_phase_parity>=1e-4 or not all(baseline_fixed_object_checks.values()):
    baseline_calibration_report["phase_parity_application_rejected"]=True
    baseline_calibration_report["candidate_native_phase_parity"]=baseline_calibration_native_phase_parity
    baseline_calibration_report["parameters_before_rebuild_gate"]=baseline_calibration_parameters.copy()
    baseline_calibration_report["accepted"]=False
    baseline_calibration_report["reason"]="Insufficient native parity or support/amplitude/relaxation/weights changed after resampling; zero parameters restored."
    baseline_calibration_parameters={name:0. for name in baseline_calibration_tools.PARAMETER_NAMES}
    baseline_calibration_report["parameters"]=baseline_calibration_parameters.copy()
    if "baseline" in baseline_calibration_report:
        baseline_calibration_report["applied"]=deepcopy(baseline_calibration_report["baseline"])
    del model
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    prepared,model=rebuild_baseline_model(baseline_calibration_parameters)
    zero_currents=np.zeros((model.nc,model.nstates))
    baseline_image,baseline_kdata=model.forward(zero_currents)
    baseline_calibration_native_phase_parity=float(np.linalg.norm(baseline_kdata-uncalibrated_baseline_kdata)
        /max(np.linalg.norm(uncalibrated_baseline_kdata),np.finfo(float).tiny))
baseline_calibration_report["applied_fixed_object_rebuild_checks"]=fixed_object_rebuild_checks(prepared)
assert (baseline_calibration_native_phase_parity<1e-4 and
        all(baseline_calibration_report["applied_fixed_object_rebuild_checks"].values())),"The restored model no longer matches the original baseline; reload the inputs."
experiment=prepared["expanded_experiment"]
calibration=prepared["expanded_calibration"]
preparation_report=prepared["report"]
X_image,Y_image=model.reconstruction.X_m,model.reconstruction.Y_m
ideal_image=model.ideal_image
baseline_model_mse=float(np.sum((abs(baseline_image)-abs(ideal_image))**2)/np.sum(abs(ideal_image)**2))
baseline_measured_metrics=compare_magnitude(measured_image,baseline_image,X_m=X_image,Y_m=Y_image)
baseline_calibration_report["native_phase_parity"]=baseline_calibration_native_phase_parity
baseline_calibration_report["native_sample_chunk"]={"uncalibrated":uncalibrated_sample_chunk,
    "requested_rebuild":uncalibrated_sample_chunk,"effective_rebuild":int(model.sample_chunk)}
baseline_calibration_report["preparation"]=deepcopy(preparation_report)
model.runtime_info["baseline_calibration"]=baseline_calibration_parameters.copy()
model.runtime_info["baseline_calibration_accepted"]=baseline_calibration_report["accepted"]
model.runtime_info["baseline_calibration_native_phase_parity"]=baseline_calibration_native_phase_parity
model.runtime_info["baseline_calibration_original_sample_chunk"]=uncalibrated_sample_chunk
# Invalidate a previous optimization when rerunning only this cell.
solution=None
image_history=None
solver_ok=False
print(baseline_calibration_report["reason"])
table(list(baseline_calibration_parameters.items()),("Model parameter","Applied value"))
if "baseline" in baseline_calibration_report:
    table([(name,baseline_calibration_report["baseline"][name]["NRMSE_percent"],
            baseline_calibration_report["applied"][name]["NRMSE_percent"])
           for name in ("fit","validation","test","full_FOV")],
          ("Pixels / gain estimated on fitting pixels only","NRMSE before [%]","NRMSE after [%]"))
    before_gain=baseline_calibration_report["baseline_gain_from_fit_pixels"]
    after_gain=(baseline_calibration_report["candidate_gain_from_fit_pixels"]
                if baseline_calibration_report["accepted"] else before_gain)
else:
    before_gain=uncalibrated_baseline_measured_metrics["gain"]
    after_gain=before_gain
table([("Native MRzero / fast-phase parity",baseline_calibration_native_phase_parity),
       ("Diagnostic baseline / zero-B0 simulation MSE",baseline_model_mse)])
print("Internal validation within one acquisition: confirm parameters on another scan before drawing conclusions about their physical origin.")
fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
for ax,image,title in zip(axes,[measured_image,before_gain*np.abs(uncalibrated_baseline_image),
                               after_gain*np.abs(baseline_image)],
                         ["Measured EPI without ACDC","MRzero before calibration","MRzero after calibration"]):
    image_physical(ax,image,X_image,Y_image,title,vmin=0,vmax=measured_image.max()*1.1)
show(fig)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
np.savez_compressed(OUTPUT_DIR/"baseline_calibration.npz",
    uncalibrated_baseline_image=uncalibrated_baseline_image,
    uncalibrated_ideal_image=uncalibrated_ideal_image,
    uncalibrated_baseline_kdata=uncalibrated_baseline_kdata,
    calibrated_baseline_image=baseline_image,calibrated_ideal_image=ideal_image,
    calibrated_baseline_kdata=baseline_kdata,measured_image=measured_image,
    **baseline_calibration_masks)
(OUTPUT_DIR/"baseline_calibration.json").write_text(
    json.dumps(json_safe(baseline_calibration_report),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
(OUTPUT_DIR/"baseline_metadata_audit.json").write_text(
    json.dumps(json_safe(baseline_metadata_audit),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")

# Then rerun the bidirectional diagnostic with this model.
baseline_signal_disagreement=None
baseline_signal_disagreement_arrays={}

### 4c — Check signal loss in both directions

Calibration MSE already compares intensities **in both directions**, including dark
measured pixels. However, the old “simulation below 50% of measurement” counts considered
only pixels where measured EPI was bright. They therefore excluded large dark measured
regions and quantified only deficits specific to the simulation.

This diagnostic now distinguishes **measured dark regions not reproduced** (simulation
too bright, red) from **simulated deficits absent in the measurement** (simulation
too dark, blue). It also reports regions where both images fall below the threshold,
without treating this as agreement: their continuous difference may remain substantial.
Ratio indicators use two fixed thresholds, 25% and 50%, and require the brighter image
to exceed 10% of the measured maximum, with the same scale for every model.
The full field of view is the primary diagnostic. A common envelope derived from the
uncalibrated zero-B0 diagnostic provides a second view of the object's interior;
filling this diagnostic mask does not change spin support, measured GRE, B0 or the loss.

Comparison gains are the calibration gains, estimated only on fitting pixels when
calibration is active; otherwise, the original evaluation gain is retained.
The same envelope and scale apply to all compared models. Validation/test blocks remain
separate. These intensity disagreements do not establish a particular physical cause,
a z gradient or irreversible signal loss. The ACDC optimizer then compares its image
with measured GRE. Model errors can therefore limit the predicted correction;
these diagnostics assess fidelity to EPI without ACDC before any current optimization.


In [ ]:
# Diagnostic only: no change to the object, field or loss function.
import signal_disagreement as signal_disagreement_tools
signal_disagreement_tools=importlib.reload(signal_disagreement_tools)
signal_reference_scale=float(measured_image.max())
signal_envelope=signal_disagreement_tools.fixed_comparison_envelope(
    np.abs(uncalibrated_ideal_image),threshold_fraction=.1)
signal_subsets={name.removesuffix("_mask"):mask for name,mask in baseline_calibration_masks.items()
                if np.any(mask)}
signal_sources={"before_calibration":(uncalibrated_baseline_image,before_gain),
                "after_calibration":(baseline_image,after_gain)}
if point_baseline_image is not None:
    signal_sources={"point_before_calibration":(point_baseline_image,before_gain),**signal_sources}
baseline_signal_disagreement={"comparison_policy":{
    "primary_domain":"full FOV, including measured-dark pixels",
    "secondary_domain":"fixed envelope from uncalibrated zero-B0 target; diagnostic only",
    "envelope_threshold_fraction":.1,"reference_scale":signal_reference_scale,
    "bright_fraction":.1,"both_low_does_not_imply_agreement":True,
    "gain_source":"calibration fit pixels only" if baseline_calibration_enabled else "original full-FOV evaluation gain",
    "point_comparison_gain":"same before-calibration gain as the uncalibrated finite-voxel model",
    "object_target_field_and_optimization_loss_unchanged":True,
    "independent_physical_validation":False},"models":{}}
baseline_signal_disagreement_arrays={"fixed_comparison_envelope_mask":signal_envelope.copy(),"X_image_m":X_image.copy(),"Y_image_m":Y_image.copy()}
signal_rows=[]
signal_energy_rows=[]
signal_after_full=None
for name,(simulated,comparison_gain) in signal_sources.items():
    baseline_signal_disagreement["models"][name]={}
    for domain,analysis_mask in (("full_FOV",None),("fixed_envelope",signal_envelope)):
        result=signal_disagreement_tools.signal_disagreement(measured_image,simulated,
            gain=comparison_gain,analysis_mask=analysis_mask,subsets=signal_subsets or None,
            reference_scale=signal_reference_scale,bright_fraction=.1,ratios=(.25,.5))
        baseline_signal_disagreement["models"][name][domain]=result["report"]
        for key,value in result["arrays"].items():
            baseline_signal_disagreement_arrays[name+"_"+domain+"_"+key]=value
        maps=result["arrays"]
        energy=result["report"]["analysis_mask"]
        signal_energy_rows.append((name,domain,energy["NRMSE_percent"],
            energy["overprediction_normalized_SSE"],energy["underprediction_normalized_SSE"],
            energy["both_low_fraction_of_SSE"]))
        signal_rows.append((name,domain,
            int(np.count_nonzero(maps["measured_low_simulated_high_50_mask"])),
            int(np.count_nonzero(maps["simulated_low_measured_high_50_mask"])),
            int(np.count_nonzero(maps["both_low_mask"]))))
        if name=="after_calibration" and domain=="full_FOV":signal_after_full=result
table(signal_energy_rows,("Model","Domain","NRMSE [%]","Normalized SSE: simulation too bright",
    "Normalized SSE: simulation too dark","SSE fraction: both below threshold"))
table(signal_rows,("Model","Domain","Measured dark / simulation too bright [pixels]",
    "Simulation dark / measured bright [pixels]","Both below threshold, agreement not guaranteed [pixels]"))
print("Legacy counts measured only simulation underprediction. Measured signal losses that are inadequately reproduced must also be examined; colored indicators do not establish their physical cause.")
print("Prioritize continuous signed errors and their SSE; descriptive indicators use 25% / 50% ratios and a bright image >= 10% of the fixed measured maximum. Both images below threshold does not establish agreement.")
maps=signal_after_full["arrays"]
classification=np.full(measured_image.shape,np.nan)
classification[maps["simulated_low_measured_high_50_mask"]]=-1.
classification[maps["measured_low_simulated_high_50_mask"]]=1.
difference=maps["difference"]
diff_max=max(float(np.nanmax(np.abs(difference))),np.finfo(float).tiny)
fig,axes=plt.subplots(1,4,figsize=(20,4),layout="constrained")
image_physical(axes[0],measured_image,X_image,Y_image,"Measured EPI without ACDC",vmin=0,vmax=signal_reference_scale)
image_physical(axes[1],after_gain*np.abs(baseline_image),X_image,Y_image,"Compared MRzero baseline",vmin=0,vmax=signal_reference_scale)
im=image_physical(axes[2],difference,X_image,Y_image,"Simulation − measured\nRed: simulation too bright",cmap="RdBu_r",vmin=-diff_max,vmax=diff_max)
fig.colorbar(im,ax=axes[2],shrink=.8)
image_physical(axes[3],measured_image,X_image,Y_image,"Red: measurement weaker than simulation\nBlue: simulation weaker (50% ratio, 10% brightness)",vmin=0,vmax=signal_reference_scale)
axes[3].pcolormesh(X_image*1000,Y_image*1000,classification,shading="auto",cmap="RdBu_r",vmin=-1,vmax=1,alpha=.8)
show(fig)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
np.savez_compressed(OUTPUT_DIR/"baseline_signal_disagreement.npz",**baseline_signal_disagreement_arrays)
(OUTPUT_DIR/"baseline_signal_disagreement.json").write_text(
    json.dumps(json_safe(baseline_signal_disagreement),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
baseline_calibration_report["native_model_signal_disagreement"]=baseline_signal_disagreement
(OUTPUT_DIR/"baseline_calibration.json").write_text(
    json.dumps(json_safe(baseline_calibration_report),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")


### Check the model before optimization

Model 09 recomputes the PDG at every evaluation, including trial currents and
verification after DAC conversion. Hardware limits use the same eight corners
as 08. This cell checks for a finite signal before starting the solver.


In [ ]:
zero_current_parity=None
reference_validation_report={
    "passed":bool(np.isfinite(baseline_kdata).all() and np.isfinite(baseline_image).all()),
    "simulator":model.runtime_info["simulator"],
    "native_backend":model.runtime_info.get("native_B0_adapter",{}),
    "validation":"Independent backend tests and the 08/09 comparison are described in README.md."}
table(list(reference_validation_report.items()),("Native MRzero check","Value"))
assert reference_validation_report["passed"]

## 5. Initialize with a static shim and check hardware limits

The static solution provides a starting point for image optimization. All subspins
participate in field limits. Unmeasured electrical limits remain disabled;
no R/L or thermal parameters are invented.


In [ ]:
if IMAGE_INITIALIZATION=="static":
    static_fit=solve_field_gpu(model.maps_numpy[model.quality_mask],
        -experiment["field"]["b0Hz"][model.quality_mask],device=model.device,dtype=model.dtype,
        max_current_A=CHANNEL_MAX_CURRENT_A,total_current_limit_A=TOTAL_CURRENT_LIMIT_A,
        field_limit_Hz=FIELD_LIMIT_HZ,max_iterations=1000,tolerance=1e-7)
    requested_initial=np.repeat(static_fit["currents_A"][:,:1],model.nstates,axis=1)
elif IMAGE_INITIALIZATION=="zero":requested_initial=zero_currents.copy()
else:raise ValueError("IMAGE_INITIALIZATION must be static or zero.")
initial_currents=model.project_currents(requested_initial,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
initial_hardware=model.hardware.diagnostics(initial_currents,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
assert initial_hardware["all_limits_met"]
initial_image,_=model.forward(initial_currents)
table(list(initial_hardware["checks"].items()),("Initial check","Result"))
print("Equivalent response: tau [µs] =",model.current_response["tau_s"][:3]*1e6)

## 6. Optimize currents with live display

The target is **the selected measured GRE scan**. Its matrix may differ from the EPI
matrix: it is interpolated onto the physical EPI grid without extrapolation.
The mask and scale are fixed from the baseline before optimization, then retained
for every current setting, display and verification after DAC quantization.

The image loss is `J_image(I) = sum_M((g*abs(EPI_MRzero(I)) - GRE_measured)**2) / sum_M(GRE_measured**2)`.
`g` is the fixed positive simulation→EPI without ACDC→GRE gain; `M` is the common
coverage (or the explicitly selected GRE ROI). Dark pixels remain penalized.
The evaluated model uses B0, losses fitted to the 32 echoes and each ACDC response
at every subspin before complex summation. The zero-B0 simulation is a separate
diagnostic and never replaces the GRE target. Orientation enters the gradients.
The live display preserves acquired pixel order; physical figures use transformed
coordinates. CUDA graphs from 03 are disabled to retain hardware projection from 04.

The **Scan current** slider lets you review images as they arrive.
Only magnitudes at iterations `0, LIVE_UPDATE_EVERY, 2*LIVE_UPDATE_EVERY...` are retained;
no current or k-space history is added. The loss curve retains scalar values at every
iteration. The last live display may therefore correspond to an unarchived iteration;
the best result remains the one in section 7.

The gallery is saved at every update in `optimization_images.npz` and
`optimization_images.html`. The HTML file also lets you browse images outside the
notebook. Its scale remains fixed and shared by all archived images.

The table below distinguishes the **compute device** from **MRzero cache storage**.
A cache in CPU RAM can feed CUDA execution in blocks; transfers may then reduce
observed GPU activity. `use_cuda_graph=False` retains hardware checks and does not
disable CUDA computation. If the model displays `cpu` while a GPU is requested,
rebuild the model after fixing the environment and restarting the kernel.

With `FIELD_GUIDANCE_WEIGHT > 0`, the solver minimizes
`J_total = J_image + FIELD_GUIDANCE_WEIGHT * mean((B0_residual / FIELD_GUIDANCE_SCALE_HZ)**2)`.
The residual is calculated at the physical positions of reliable subspins by averaging
the actual ACDC field between temporal centers of successive EPI lines.
Exact amplifier-response integrals are used. B0, its mean and large offsets remain
in Hz: no reduction modulo the FOV or modulo 2π is applied.
Fourier reconstruction already represents aliasing; this guidance helps select
the physical correction when several aliased images look similar.

`0.05` is an initial weight to evaluate, not a setting validated for every phantom.
`0` restores image-only behavior. Displayed MSE and NRMSE remain strictly image metrics;
the `objective`, `field_penalty` and `residual_field_rms_Hz` histories record guidance
separately. The best result is selected using total loss, which may differ from the
lowest image MSE encountered. In guided mode, `TARGET_MSE` is diagnostic only:
reaching low MSE is insufficient to stop the solver on an incorrect aliasing branch.

Guidance assumes a correctly unwrapped B0 map **with the correct absolute frequency
reference**, using the same scanner frequency as EPI. It does not exceed any current
or field limit and keeps disabled channels at zero. It guarantees neither perfect
correction beyond coil capacity nor recovery of lost signal.
Rerun from the parameters after changing the weight.


In [ ]:
# Install the measured target AFTER calibrating/rebuilding the model.
optimization_target=configure_measured_gre_target(model,inputs["gre"],
    baseline_image=baseline_image,measured_baseline_image=measured_image,
    frame_matrix=O,threshold_fraction=OPTIMIZATION_REFERENCE_THRESHOLD)
optimization_reference_image=optimization_target["reference_image"]
optimization_reference_mask=optimization_target["mask"]
optimization_target_report=optimization_target["report"]
table(list(optimization_target_report.items()),("GRE optimization target","Value"))
table([("Baseline / measured GRE MSE",relative_image_mse(model,baseline_image)),
       ("Initialization / measured GRE MSE",relative_image_mse(model,initial_image))])
fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
target_vmax=max(float(optimization_reference_image.max()),float(np.abs(baseline_image).max()))
for ax,image,title in zip(axes,[optimization_reference_image,baseline_image,initial_image],
                         ["Target: measured GRE (fixed scale)","EPI simulation without ACDC","Initial EPI simulation"]):
    image_physical(ax,np.where(optimization_reference_mask,np.abs(image),np.nan),
        X_image,Y_image,title,vmin=0,vmax=target_vmax)
show(fig)

solution=None
image_history=None
if RUN_OPTIMIZATION:
    if GPU_REQUIRED and model.device.type!="cuda":
        raise RuntimeError("Optimization requires CUDA, but the current model is on the CPU. "
            "Fix the PyTorch/CUDA environment, restart the kernel and rerun from the beginning. "
            "Changing GPU_PROFILE alone does not move an existing model.")
    table([("Optimization compute device",str(model.device)),
           ("Model tensors",str(model.B.device)),
           ("Precision",str(model.dtype)),
           ("GPU",model.runtime_info["gpu_name"]),
           ("MRzero cache storage",model.runtime_info["native_baseline_cache"]),
           ("ADC samples per block",model.sample_chunk)])
    image_history=OptimizationImageHistory(optimization_reference_image,baseline_image,every=LIVE_UPDATE_EVERY,
        title="Scan current — optimization images",origin="upper")
    image_history_npz=OUTPUT_DIR/"optimization_images.npz"
    image_history_html=OUTPUT_DIR/"optimization_images.html"
    live=LiveOptimizationPlot(reference=optimization_reference_image,baseline=baseline_image,
        current_limit_a=MAX_CURRENT_A,roi_mask=optimization_reference_mask,
        title=f"09: alignment {ALIGNMENT_ANGLE_DEG}°, mirrors X={ALIGNMENT_MIRROR_X}, Y={ALIGNMENT_MIRROR_Y}")
    def update(snapshot):
        live.set_history(snapshot["history"]["iteration"],snapshot["history"]["fval"])
        live.update(iteration=snapshot["iteration"],image=snapshot["image"],currents_a=snapshot["currents_A"],
            loss=snapshot["fval"],nrmse_percent=100*np.sqrt(snapshot["fval"]),
            max_field_hz=snapshot["max_field_Hz"],field_limit_hz=FIELD_LIMIT_HZ)
        if image_history.record(snapshot):
            image_history.save_npz(image_history_npz)
            image_history.write_html(image_history_html)
            image_history.display()
    try:
        solution=optimize_mrzero_image(model,initial_currents,total_current_limit_A=TOTAL_CURRENT_LIMIT_A,
            field_limit_Hz=FIELD_LIMIT_HZ,max_iterations=MAX_ITERATIONS,learning_rate_A=LEARNING_RATE_A,
            target_mse=TARGET_MSE,patience=PATIENCE,callback=update,callback_every=LIVE_UPDATE_EVERY,
            use_cuda_graph=False,field_guidance_weight=FIELD_GUIDANCE_WEIGHT,
            field_guidance_scale_Hz=FIELD_GUIDANCE_SCALE_HZ)
    finally:
        live.finish("Finished" if solution is not None else "Stopped: last evaluated state")
    info=solution["optimization_info"]
    currents=solution["currents_A"]
    history=solution["history"]
    table(list(info.items()))
    if "field_guidance" in info:
        guide_report=info["field_guidance"]
        table([("Initial total loss",solution["initial_objective"]),("Selected total loss",info["objective"]),
               ("Initial residual B0 RMS [Hz]",guide_report["initial_residual_field_rms_Hz"]),
               ("Selected residual B0 RMS [Hz]",guide_report["final_residual_field_rms_Hz"])])
    print("Image gallery:",image_history_html)
else:print("Optimization disabled; model diagnostics remain available.")


## 7. Check physical currents and DAC conversion

Limits are recalculated between ADCs, over the complete TR, and after nominal
DAC conversion. Quantized MSE must also improve on the initial point.
In guided mode, total loss after DAC must also be no greater than the initial
total loss. The weight and scale come from the saved result, rather than a parameter
cell that may have changed after execution.
Reaching the iteration limit is reported as such, without claiming mathematical convergence.


In [ ]:
solver_ok=False
if solution is not None:
    hardware_report=model.hardware.diagnostics(currents,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
    quantized_currents=quantize_currents(currents)
    quantized_hardware=model.hardware.diagnostics(quantized_currents,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
    quantized_image,quantized_kdata=model.forward(quantized_currents)
    quantized_mse=relative_image_mse(model,quantized_image)
    quantized_objective=quantized_mse
    quantized_field_guidance=None
    if info.get("field_guidance_weight",0.)>0:
        dac_guide=FieldGuidance(model,scale_Hz=info["field_guidance_scale_Hz"])
        dac_penalty,_,dac_rms=dac_guide.value_gradient(model._check_currents(quantized_currents),gradient=False)
        quantized_objective+=info["field_guidance_weight"]*float(dac_penalty)
        quantized_field_guidance={"field_penalty":float(dac_penalty),"residual_field_rms_Hz":float(dac_rms)}
    solver_ok=bool(hardware_report["all_limits_met"] and quantized_hardware["all_limits_met"]
        and np.isfinite(info["fval"]) and info["fval"]<=solution["initial_relative_mse"]+1e-7
        and np.isfinite(quantized_mse) and quantized_mse<=solution["initial_relative_mse"]+1e-7
        and np.isfinite(quantized_objective) and quantized_objective<=solution["initial_objective"]+1e-7)
    table([(key,value,quantized_hardware["checks"][key]) for key,value in hardware_report["checks"].items()],
          ("Hardware check","Optimized","After DAC"))
    table([("Initial-point MSE",solution["initial_relative_mse"]),("Optimized MSE",info["fval"]),
           ("MSE after DAC",quantized_mse),("Numerical export checks",solver_ok)])
    if quantized_field_guidance is not None:
        table([("Total loss after DAC",quantized_objective),
               ("Residual B0 RMS after DAC [Hz]",quantized_field_guidance["residual_field_rms_Hz"])])
    fig,axes=plt.subplots(1,4,figsize=(18,4),layout="constrained")
    vmax=max(abs(image).max() for image in (optimization_reference_image,baseline_image,solution["image"],quantized_image))
    for ax,image,title in zip(axes,[optimization_reference_image,baseline_image,solution["image"],quantized_image],
                             ["Measured GRE target","Without ACDC","Optimized ACDC","After DAC"]):
        image_physical(ax,abs(image),X_image,Y_image,title,vmin=0,vmax=vmax)
    show(fig)
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
    im=axes[0].imshow(currents,cmap="RdBu_r",aspect="auto",vmin=-MAX_CURRENT_A,vmax=MAX_CURRENT_A)
    fig.colorbar(im,ax=axes[0]);axes[0].set(title="DAC commands",xlabel="Chronological line",ylabel="Channel")
    axes[1].plot(history["iteration"],history["fval"],label="Evaluated")
    axes[1].plot(history["iteration"],history["best_fval"],label="Lowest image loss encountered")
    axes[1].set(title="Relative MSE",xlabel="Iteration");axes[1].legend()
    axes[2].bar(np.arange(model.nc)+1,hardware_report["avg_abs_current_A"])
    axes[2].set(title="Mean physical current over TR",xlabel="Channel",ylabel="Mean |I| [A]")
    show(fig)
    if "objective" in history:
        fig,axes=plt.subplots(1,2,figsize=(11,4),layout="constrained")
        axes[0].plot(history["iteration"],history["objective"],label="Evaluated")
        axes[0].plot(history["iteration"],history["best_objective"],label="Best total loss")
        axes[0].set(title="Image + B0 guidance",xlabel="Iteration");axes[0].legend()
        axes[1].plot(history["iteration"],history["residual_field_rms_Hz"])
        axes[1].set(title="Physical residual field",xlabel="Iteration",ylabel="RMS [Hz]")
        show(fig)


## 8. Temporal response and subspin fields

The instantaneous/filtered comparison uses exactly the same commands.
The parent maps below are averages for display; phases and limits were calculated
for every child.


In [ ]:
if solution is not None:
    response=solution["response_diagnostics"]
    instantaneous_image,_=model.forward(currents,response_enabled=False)
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
    for ax,image,title in zip(axes,[instantaneous_image,solution["image"],abs(instantaneous_image)-abs(solution["image"])],
                             ["Same command: instantaneous response","Filtered response, 133 µs","Magnitude difference"]):
        image_physical(ax,abs(image) if "difference" not in title else image,X_image,Y_image,title,
                       cmap="RdBu_r" if "difference" in title else "gray")
    show(fig)
    field_parent=response["generated_field_at_line_center_Hz"]
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
    for ax,index in zip(axes,[0,model.nstates//2,model.nstates-1]):
        im=image_physical(ax,field_parent[...,index],prepared["physical_X_m"],prepared["physical_Y_m"],
                          f"Mean ACDC field: line {index+1} [Hz]",cmap="RdBu_r")
        fig.colorbar(im,ax=ax,shrink=.8)
    show(fig)

## 9. Save the complete optimization and orientations

Saved data include Q/P/O matrices, mirrors, patient-space subspin positions,
anchored amplitude, B0, rates, currents and checks.
Session names avoid overwriting previous optimizations.
Periodic images are stored in a separate archive; only initial and final currents are saved.


In [ ]:
saved_guidance_configuration=(
    {"weight":solution["optimization_info"]["field_guidance_weight"],
     "scale_Hz":solution["optimization_info"]["field_guidance_scale_Hz"]}
    if solution is not None else
    {"weight":FIELD_GUIDANCE_WEIGHT,"scale_Hz":FIELD_GUIDANCE_SCALE_HZ})
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
arrays={"echo_times_s":inputs["echo_times_s"],
    "object_support":fit["object_support"],"noise_mask":fit["noise_mask"],
    "noise_covariance":fit["noise_covariance"],
    "support_snr":fit["support_snr"],"support_debiased_amplitude":fit["support_debiased_amplitude"],
    "fit_valid_mask":fit["fit_valid_mask"],"target_reliable_mask":fit["target_reliable_mask"],
    "R2star_sInv":fit["R2star_sInv"],"R2star_std_sInv":fit["R2star_std_sInv"],
    "ideal_image":ideal_image,"baseline_image":baseline_image,"measured_image":measured_image,
    "initial_currents_A":initial_currents,"X_image_m":X_image,"Y_image_m":Y_image,
    "field_guidance_weight":np.asarray(saved_guidance_configuration["weight"]),
    "field_guidance_scale_Hz":np.asarray(saved_guidance_configuration["scale_Hz"]),
    "channel_max_current_A":CHANNEL_MAX_CURRENT_A,
    "channel_map":np.asarray(WAVEFORM_PARAMETERS["channel_map"],dtype=int),
    "disabled_physical_channels":np.asarray(WAVEFORM_PARAMETERS["disabled_channels"],dtype=int),
    "alignment_matrix":Q,"passive_frame_matrix":P,"combined_frame_matrix":O,
    "native_X_m":prepared["native_X_m"],"native_Y_m":prepared["native_Y_m"],
    "native_amplitude":prepared["native_amplitude"],"native_rate_sInv":prepared["native_rate_sInv"],
    "native_B0_Hz":prepared["native_B0_Hz"],"parent_valid_mask":prepared["parent_valid_mask"],
    "parent_flat_index":prepared["parent_flat_index"],"subspin_weights":prepared["weights"],
    "subspin_xyz_patient_m":prepared["xyz_m"],"subspin_B0_Hz":prepared["subspin_b0"],
    "subspin_basis_Hz_per_A":prepared["fieldbasis"],"subspin_amplitude_at_reference":experiment["m0_proxy"],
    "kx_mInv":experiment["acquisition"]["kx_mInv"],"ky_mInv":experiment["acquisition"]["ky_mInv"],
    "ADC_time_s":experiment["acquisition"]["time_s"]}
arrays.update(optimization_reference_image=optimization_reference_image,
    optimization_reference_mask=optimization_reference_mask,
    optimization_reference_measured_magnitude=optimization_target["measured_reference_magnitude"],
    optimization_reference_coverage_mask=optimization_target["coverage_mask"],
    optimization_fixed_gain=np.asarray(optimization_target["gain"]),
    measured_GRE_native_magnitude=inputs["gre"]["magnitude"],
    measured_GRE_native_X_m=inputs["gre"]["X_m"],measured_GRE_native_Y_m=inputs["gre"]["Y_m"],
    zero_b0_diagnostic_image=ideal_image)
arrays.update(uncalibrated_baseline_image=uncalibrated_baseline_image,
    uncalibrated_ideal_image=uncalibrated_ideal_image,uncalibrated_baseline_kdata=uncalibrated_baseline_kdata,
    original_native_X_m=prepared["original_native_X_m"],original_native_Y_m=prepared["original_native_Y_m"],
    native_measured_B0_Hz=prepared["native_measured_B0_Hz"],
    ADC_phase_offset_rad=experiment["acquisition"].get("phase_offset_rad",np.zeros_like(experiment["acquisition"]["time_s"])),
    **{"baseline_calibration_"+key:value for key,value in baseline_calibration_masks.items()})
arrays.update({"signal_disagreement_"+key:value for key,value in baseline_signal_disagreement_arrays.items()})
if point_baseline_image is not None:
    arrays["point_baseline_image"]=point_baseline_image
if effective_simulation_model=="finite_voxel":
    for name in ("edge_vectors_m","voxel_edge_vectors_m","b0_gradient_Hz_per_pixel","field_gradient_Hz_per_A_per_pixel"):
        arrays["voxel_"+name]=experiment["inplane_voxel"][name]
arrays["voxel_field_at_corners_Hz_per_A"]=experiment["inplane_voxel"]["field_at_corners_Hz_per_A"]
if solution is not None:
    arrays.update(optimized_image=solution["image"],optimized_kdata=solution["kdata"],currents_A=currents,
        quantized_currents_A=quantized_currents,quantized_image=quantized_image,
        iteration=history["iteration"],relative_mse=history["fval"],best_relative_mse=history["best_fval"])
    for key in ("objective","best_objective","field_penalty","residual_field_rms_Hz"):
        if key in history:arrays["optimization_"+key]=history[key]
if complex_echo_enabled:
    ce=fit["multi_echo"]
    arrays.update(complex_echo_width_Hz=ce["unresolved_width_Hz"],
        complex_echo_dispersion_kind=ce["dispersion_kind"],
        complex_echo_residual_R2_sInv=ce["residual_R2_sInv"],
        complex_echo_envelope_at_reference=ce["amplitude_envelope_at_reference"],
        complex_echo_refined_B0_Hz=ce["refined_b0_Hz"],
        complex_echo_predicted_RSS=ce["predicted_RSS"],
        complex_echo_hole_mask=complex_echo_diagnostics["hole_mask"],
        complex_echo_known_gradient_Hz_per_pixel=ce["known_inplane_gradient_Hz_per_pixel"],
        complex_echo_support_mask=ce["support_mask"],complex_echo_quality_mask=ce["quality_mask"],
        complex_echo_reduced_chi2=ce["reduced_chi2"],
        complex_echo_training_relative_RMSE=ce["training_relative_complex_RMSE"],
        complex_echo_validation_relative_RMSE=ce["validation_relative_complex_RMSE"],
        complex_echo_receiver_amplitude=ce["complex_receiver_amplitude"])
if complex_echo_enabled:
    for key in ("reference_amplitude_ratio","reference_amplitude_coverage_mask",
                "reference_contrast_field","reference_contrast_coverage_mask"):
        if key in fit["multi_echo"]:
            arrays[key]=fit["multi_echo"][key]
np.savez_compressed(OUTPUT_DIR/"mrzero09_optimization.npz",**arrays)
source_hashes=dict(inputs["source_sha256"],calibration=preparation_report["calibration_source_sha256"])
summary={"notebook":NOTEBOOK_NAME,"generation":GENERATION,
    "first_simulation":first_simulation,
    "sequence_assessment":sequence_assessment,
    "input_selection":{name:str(path.resolve()) for name,path in inputs["paths"].items()},
    "reference_sequence_validation":inputs.get("reference_sequence_validation"),
    "reference_geometry_checks":inputs.get("geometry_checks"),
    "reference_amplitude_prior":fit["report"].get("reference_amplitude_prior"),
    "optimization_target":optimization_target_report,
    "field_guidance_configuration":saved_guidance_configuration,
    "zero_b0_image_role":"Diagnostic only; optimization target is the selected measured GRE.",
    "multi_echo_fit":fit["report"].get("multi_echo"),
    "multi_echo_comparison":None if complex_echo_diagnostics is None else complex_echo_diagnostics["rows"],
    "echo_input_configuration":{"B0_sequence":B0_SEQUENCE_NAME,"B0_scan":inputs["paths"]["b0"],
        "field_echo_count":inputs["field_echo_count"],"rate_mode":effective_rate_mode,"echo_signal_model":ECHO_SIGNAL_MODEL,
        "object_amplitude_source":OBJECT_AMPLITUDE_SOURCE,
        "object_contrast_sigma_mm":OBJECT_CONTRAST_SIGMA_MM,
        "optimization_reference_threshold":OPTIMIZATION_REFERENCE_THRESHOLD,
        "reference_prior_min_validation_improvement":REFERENCE_PRIOR_MIN_VALIDATION_IMPROVEMENT,
        "object_threshold_fraction":OBJECT_THRESHOLD_FRACTION,"object_min_snr":OBJECT_MIN_SNR,
        "background_corner_m":BACKGROUND_CORNER_M,"require_noise_calibration":REQUIRE_NOISE_CALIBRATION},
    "input_dir":INPUT_DIR,"calibration_path":CALIBRATION_PATH.resolve(),
    "waveform_output_dir":WAVEFORM_OUTPUT_DIR.resolve(),"baseline_scan":inputs["paths"]["scan"].name,"orientation":angles,
    "alignment_matrix":Q,"passive_frame_matrix":P,"combined_frame_matrix":O,
    "preparation":preparation_report,"echo_fit":fit["report"],"runtime":model.runtime_info,
    "fidelity_configuration":{"requested_model":SIMULATION_MODEL,"effective_model":effective_simulation_model,
        "requested_B0_weighting":B0_WEIGHTING,"effective_B0_weighting":effective_b0_weighting},
    "baseline_signal_disagreement":baseline_signal_disagreement,
    "baseline_calibration":baseline_calibration_report,"baseline_metadata_audit":baseline_metadata_audit,
    "uncalibrated_baseline_measured_comparison":uncalibrated_baseline_measured_metrics,
    "uncalibrated_baseline_vs_zeroB0_mse":uncalibrated_baseline_model_mse,
    "fidelity_comparison_stage":"before baseline calibration",
    "fidelity_comparison":fidelity_comparison,"point_baseline_measured_comparison":point_baseline_metrics,
    "zero_current_validation":reference_validation_report,
    "zero_current_reference":reference_validation_report,
    "zero_current_V4_parity":zero_current_parity,"baseline_measured_comparison":baseline_measured_metrics,
    "baseline_vs_zeroB0_mse":baseline_model_mse,"current_limits_A":{"per_channel":CHANNEL_MAX_CURRENT_A,"total":TOTAL_CURRENT_LIMIT_A},
    "current_response":CURRENT_RESPONSE,"hardware_configuration":HARDWARE,
    "disabled_channels":WAVEFORM_PARAMETERS["disabled_channels"],
    "waveform_parameters":WAVEFORM_PARAMETERS,
    "source_sha256":source_hashes,"export_checks_passed":solver_ok,
    "image_history":{"every":LIVE_UPDATE_EVERY,
        "iterations":image_history.arrays()["iterations"] if image_history is not None else [],
        "archive":"optimization_images.npz" if image_history is not None else None,
        "viewer":"optimization_images.html" if image_history is not None else None}}
summary["waveform_adc_export"]=globals().get("adc_waveform_export")
if solution is not None:
    summary.update(optimization=info,hardware=hardware_report,hardware_after_DAC=quantized_hardware,
        quantized_relative_mse=quantized_mse,initial_relative_mse=solution["initial_relative_mse"],
        initial_objective=solution["initial_objective"],quantized_objective=quantized_objective,
        quantized_field_guidance=quantized_field_guidance)
(OUTPUT_DIR/"mrzero09_optimization.json").write_text(json.dumps(json_safe(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
print("Results saved:",OUTPUT_DIR)

### 9b. Static and dynamic field-shim waveforms

Both field fits follow the methods in the
[ACDC_Optim](https://github.com/bughht/ACDC_Optim) repository. They use the same
calibration, frame, hardware constraints and synchronization as the image CSV.
They are calculated even when `RUN_OPTIMIZATION=False` or
`IMAGE_INITIALIZATION="zero"`. Files are retained in `INPUT_DIR/waveform`,
with their parameters and checks.


In [ ]:
field_shim_exports=None
if EXPORT_FIELD_SHIM_WAVEFORMS:
    field_shim_exports=export_field_shim_waveforms(
        model,WAVEFORM_OUTPUT_DIR,WAVEFORM_PARAMETERS,
        total_current_limit_A=TOTAL_CURRENT_LIMIT_A,field_limit_Hz=FIELD_LIMIT_HZ,
        timing=inputs["timing"],sequence_path=inputs["paths"]["scan_sequence"],
        csv_time_origin=CSV_TIME_ORIGIN,trigger_channel=TRIGGER_CHANNEL,
        raster_s=FIELD_SHIM_RASTER_S,max_iterations=FIELD_SHIM_MAX_ITERATIONS,
        provenance=globals().get("source_hashes",{}))
    summary["field_shim_waveforms"]=field_shim_exports
    table([(name,entry["csv_file"]) for name,entry in field_shim_exports["exports"].items()],
          ("Shimming method","Saved CSV"))
else:
    print("Static and dynamic exports disabled.")


## 10. Optional CSV export, synchronized to the selected origin

The ext1 trigger in the first imaging block is checked in the `.seq` file.
In `first_imaging_trigger` mode, trigger time is subtracted from the RF-relative
times used by the simulation. The CSV retains times in seconds and a return to zero.
It contains amperes, using the channel/polarity mapping selected at the start.
The notebook creates a local file; it does not program the controller.

All three methods export into `INPUT_DIR/waveform`: `waveformN.csv` for image
optimization, and `waveform_static.csv` and `waveform_dynamic.csv` for field shims.
Rerunning adds a number and retains previous files. Scientific results and figures
remain in `OUTPUT_DIR`.

For image optimization, initial states with all currents zero are removed,
then the first nonzero command time is subtracted from the remaining times:
that first command is at `t=0`, without adding a row.
Intervals between commands and the final return to zero are retained.
The offset relative to the verified origin is recorded in metadata and used
to overlay the CSV on the model.

Each image-optimization CSV also has a companion `waveformN_zero_time.csv`:
the same rows, currents and order, with only the first column replaced by zeros.


In [ ]:
trigger_report=sequence_trigger_timing(inputs["paths"]["scan_sequence"],inputs["timing"],channel=TRIGGER_CHANNEL)
csv_edges,csv_timing=waveform_edges_for_origin(model,origin=CSV_TIME_ORIGIN,
    trigger_rf_s=trigger_report["trigger_rf_s"],min_update_interval_s=8e-6)
summary["trigger"]=trigger_report
summary["CSV_timing"]=csv_timing
print("Origin before time shift:",CSV_TIME_ORIGIN,"| bounds [ms]:",csv_edges[[0,-1]]*1000)
csv_file=None
csv_zero_time_file=None
if EXPORT_WAVEFORM and solution is not None and solver_ok:
    export=export_waveform(currents,csv_edges,WAVEFORM_PARAMETERS,WAVEFORM_OUTPUT_DIR,
        include_header=True,include_zero_preamble=True,include_final_zero_row=True,
        zero_time_column=False,numbered_filename=True,start_at_first_nonzero=True)
    csv_file=Path(export["csv_file"])
    summary["waveform_export"]=export
    print("Shift to the first nonzero command [ms]:",export["time_origin_shift_s"]*1000,
          "| initial zero states removed:",export["leading_zero_states"])
    zero_time_export=export_zero_time_copy(csv_file)
    csv_zero_time_file=Path(zero_time_export["csv_file"])
    summary["waveform_zero_time_export"]=zero_time_export
    print("CSV with zero time column:",csv_zero_time_file)
    print("CSV:",csv_file)
else:print("CSV export disabled, or numerical checks failed.")
(OUTPUT_DIR/"mrzero09_optimization.json").write_text(json.dumps(json_safe(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")

## 11. Display the final ACDC waveform

DAC commands and predicted filtered currents are plotted for every output,
with a zoom on the largest command change. `WAVEFORM_PREVIEW_CHANNEL` selects
a specific output. The return to zero and final decay are included.

Output numbers, polarities and times follow the CSV settings. If a CSV was created,
its values are reread to display the commands actually saved.
The filtered current is a model prediction with a zero initial state.

The full exported CSV path is displayed before the figure.


In [ ]:
if csv_file is not None:
    print("Saved CSV:",Path(csv_file).resolve())
else:
    print("No CSV exported for this session.")
    print("Planned output folder:",WAVEFORM_OUTPUT_DIR.resolve())

waveform_report=None
if solution is not None:
    csv_origin_rf_s=trigger_report["trigger_rf_s"] if CSV_TIME_ORIGIN=="first_imaging_trigger" else 0.0
    csv_export=summary.get("waveform_export",{}) if csv_file is not None else {}
    csv_origin_rf_s+=csv_export.get("time_origin_shift_s",0.0)
    fig,waveform_report=plot_acdc_waveform(model,currents,
        origin=CSV_TIME_ORIGIN,trigger_rf_s=trigger_report["trigger_rf_s"],
        channel_map=WAVEFORM_PARAMETERS["channel_map"],
        channel_polarity=WAVEFORM_PARAMETERS["channel_polarity"],
        channel=WAVEFORM_PREVIEW_CHANNEL,csv_file=csv_file,csv_origin_rf_s=csv_origin_rf_s,
        csv_leading_zero_states=csv_export.get("leading_zero_states",0))
    fig.savefig(OUTPUT_DIR/"mrzero09_acdc_waveform.png",dpi=150)
    show(fig)
    summary["waveform_preview"]=waveform_report
    (OUTPUT_DIR/"mrzero09_optimization.json").write_text(
        json.dumps(json_safe(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
else:print("Run optimization to display the final ACDC waveform.")


## 12. Compare measured acquisitions in the `results` folder

If `INPUT_DIR/results` contains scans, this cell displays **reference GRE /
EPI without ACDC / corrected EPI** for each one. `CORRECTED_SCAN_NAMES=None` uses
all `*scan.dat` files; a list of names selects specific acquisitions.
Their names appear on the figures to identify each measured result.

As in 04, one gain is calculated from EPI **without ACDC**, then applied to every
corrected scan. GRE normalization, mask and grayscale are shared.
The ROI uses `COMPARISON_REFERENCE_THRESHOLD` times the GRE maximum; use `0`
for the entire common coverage. Images are reconstructed per receiver before RSS,
then compared in native physical coordinates. Slice centers, orientations and
sequence parameters are checked.

Reference GRE is interpolated onto the EPI grid of the selected sequence.
Rotation and mirrors apply to common display coordinates. RMSE and NRMSE describe
measured images; they remain separate from the MSE used to optimize the MRzero twin.
The correspondence between an acquisition and the current CSV is not inferred
from its filename.

Files incompatible with the supplied sequence or slice are listed in a table,
then excluded.

A file that cannot be opened is reported and skipped for this comparison.
Once the file becomes accessible, rerun only the comparison cell.


In [ ]:
# Reload the fix without rerunning the simulation steps.
import importlib
import measured_scan_comparison as measured_comparison_tools
compare_measured_scan_set=importlib.reload(measured_comparison_tools).compare_measured_scan_set
measured_comparison=None
results_dir=INPUT_DIR/"results"
if not results_dir.is_dir():
    print("No results folder: measured comparison skipped.")
else:
    corrected_paths=(sorted(results_dir.glob("*scan.dat")) if CORRECTED_SCAN_NAMES is None
                     else [results_dir/name for name in CORRECTED_SCAN_NAMES])
    if not corrected_paths:
        print("No corrected scan selected in",results_dir)
    else:
        measured_comparison=compare_measured_scan_set(inputs,corrected_paths,frame_matrix=O,
            normalize=NORMALIZE_COMPARISON,reference_threshold_fraction=COMPARISON_REFERENCE_THRESHOLD,
            skip_incompatible=True)
        comparison=measured_comparison
        report=comparison["report"]
        if report["skipped_scans"]:
            table([(item["filename"],item["reason"]) for item in report["skipped_scans"]],
                  ("File not compared","Reason"))
        if not comparison["images"]:
            print("No readable corrected scan matches the sequence and reference slice.")
        else:
            units=report["RMSE_units"]
            table([("Shared gain: without ACDC → GRE",f"{comparison['gain']:.6g}"),
                   ("GRE normalization scale",f"{comparison['reference_scale']:.6g}"),
                   ("RMSE units",units),
                   ("Common-mask pixels",report["common_pixels"]),
                   ("Pixels excluded for missing coverage",report["excluded_missing_coverage_pixels"])])
            table([(row["label"],f"{row['RMSE']:.6f}",f"{row['NRMSE_percent']:.3f}")
                   for row in comparison["rows"]],("Measured scan vs GRE","RMSE","NRMSE [%]"))
            vmax=max(float(np.nanmax(image)) for image in comparison["images"])
            baseline_row=comparison["rows"][0]
            for index,path in enumerate(map(Path,report["corrected_scan_paths"])):
                corrected_row=comparison["rows"][index+1]
                panels=[comparison["images"][0],comparison["images"][1],comparison["images"][index+2]]
                titles=["Reference GRE",f"Scan without ACDC\nRMSE = {baseline_row['RMSE']:.4f}",
                        f"Scan with ACDC\n{path.name}\nRMSE = {corrected_row['RMSE']:.4f}"]
                fig,axes=plt.subplots(1,3,figsize=(18,5),layout="constrained",sharex=True,sharey=True)
                for ax,image,title in zip(axes,panels,titles):
                    ax.set_facecolor("black")
                    im=image_physical(ax,image,comparison["X_m"],comparison["Y_m"],title,vmin=0,vmax=vmax)
                fig.colorbar(im,ax=axes,label=f"Magnitude [{units}]")
                fig.savefig(OUTPUT_DIR/f"mrzero09_measured_comparison_{index+1}.png",dpi=150)
                show(fig)
            np.savez_compressed(OUTPUT_DIR/"measured_comparison.npz",
                images=np.stack(comparison["images"]),labels=np.asarray(comparison["labels"]),
                mask=comparison["mask"],X_m=comparison["X_m"],Y_m=comparison["Y_m"],
                gain=comparison["gain"],reference_scale=comparison["reference_scale"])
        comparison_summary=dict(report,metrics=comparison["rows"])
        (OUTPUT_DIR/"measured_comparison.json").write_text(
            json.dumps(json_safe(comparison_summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
        summary["measured_comparison"]=comparison_summary
        (OUTPUT_DIR/"mrzero09_optimization.json").write_text(
            json.dumps(json_safe(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")